# Notebook 19 — Statistical power, mitigation, and label-efficiency for the NCA resubmission

Single RunPod notebook closing the gaps that made the previous submission a diagnosis-only paper.
It **extends** Notebook 18 rather than replacing it: identical normalization key, identical
overlap filter, identical metric functions, identical split files. New rows merge with NB18's
90 rows to form one 10-seed record.

## What this notebook adds

| Stage | Experiment | Purpose |
|---|---|---|
| **A** | Single-source matrix at 10 seeds (trains the 5 new seeds only) | Exact sign-flip test can now reach `p = 0.00195`; the "cannot attain p < 0.05" sentence is deleted from the paper |
| **B** | Leave-one-corpus-out + pooled-all training | First mitigation. LOCO is the deployment-realistic protocol and becomes the headline table |
| **C** | Target label-efficiency curves | **The practical payload.** How many labelled target examples recover 90% of target-trained macro-F1 |
| **D** | DANN (gradient-reversal) on the LOCO folds | One standard domain-generalization comparator |
| **E** | Source-validation threshold recalibration | CPU-only. Attempts to fix the sarcastic-class collapse |
| **F** | Near-duplicate sensitivity | CPU-only. Re-scores saved predictions on a stricter target subset; defends the deduplication claim without retraining |
| **G** | Local open-weight LLM baseline (optional) | Removes the dataset-licensing objection to the prompted-API experiment |

## Scientific contract

- Primary architecture is unchanged: `csebuetnlp/banglabert` with its standard sequence-classification head.
- Checkpoint selection and temperature fitting use **source validation only**. Target test data are
  never consulted for any selection decision.
- Few-shot target examples in Stage C are drawn from the target **train** split only.
- The configuration is hashed and the git commit recorded **before** any training runs
  (`19_run_config.json`), so the confirmatory runs are auditably frozen.
- Every stage is independently gated and resumable. A stage that fails leaves prior stages saved.

## Outputs

Tables → `04_outputs/finalized_outputs/tables/`
Figures → `04_outputs/finalized_outputs/figures/` (vector PDF + 600-dpi PNG)
Predictions → `04_outputs/predictions/19_*/`

## Before running

1. Confirm the pod's hourly rate on the RunPod pod page and set `RUNPOD_GPU_HOURLY_USD`.
2. Set `NB19_MAX_COMPUTE_USD` to the amount you are willing to spend. The budget guard raises
   before starting a job that would exceed it.
3. Community Cloud RTX 4090 is the right instance. Secure Cloud costs roughly double for no benefit here.

In [1]:
# Same tested stack as Notebook 18. Restart the kernel only if pip explicitly requests it.
%pip install -q "transformers>=4.44,<5" "accelerate>=0.33" "safetensors>=0.4" \
    "sentencepiece>=0.2" "scikit-learn>=1.3" "scipy>=1.10" "pandas>=2.0" \
    "matplotlib>=3.7" "nbformat>=5.9"

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -q hf_transfer

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os, re, gc, json, math, time, random, shutil, hashlib, platform, subprocess, warnings, unicodedata, itertools
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import scipy
from scipy import stats
from scipy.optimize import minimize_scalar
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)
import transformers

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def find_repo_root():
    env = os.getenv("NB19_ROOT", os.getenv("NB18_ROOT", "")).strip()
    candidates = ([Path(env)] if env else []) + [
        Path("/workspace/Sarcasm_detection"), Path.cwd(), *Path.cwd().resolve().parents
    ]
    for c in candidates:
        if (c / "01_data" / "interim" / "splits").exists():
            return c.resolve()
    raise RuntimeError("Repository root not found: need 01_data/interim/splits/.")

ROOT    = find_repo_root()
SPLITS  = ROOT / "01_data" / "interim" / "splits"
CKPT    = ROOT / "03_checkpoints" / "19_ncaa"
OUT     = ROOT / "04_outputs"
TABLES  = OUT / "tables"
PRED    = OUT / "predictions"
FINAL   = OUT / "finalized_outputs"
FT, FF  = FINAL / "tables", FINAL / "figures"
for p in (CKPT, TABLES, PRED, FT, FF):
    p.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "csebuetnlp/banglabert"
CORPORA = ["ben_sarc_binary", "banglasarc_binary", "banglasarc3_binary"]
DISPLAY = {"ben_sarc_binary": "Ben-Sarc", "banglasarc_binary": "BanglaSarc",
           "banglasarc3_binary": "BanglaSarc3"}
SYSTEMS = {"vanilla": {"adversarial": None, "fgm_epsilon": None},
           "fgm":     {"adversarial": "fgm", "fgm_epsilon": 0.5}}

# Seeds 42/1/7/123/2024 were completed by Notebook 18. Stage A trains only the new five.
SEEDS_NB18 = [42, 1, 7, 123, 2024]
SEEDS_NEW  = [2025, 13, 77, 314, 1337]
SEEDS_10   = SEEDS_NB18 + SEEDS_NEW

LOCO_SEEDS      = [42, 1, 7, 123, 2024]      # 5 seeds; the pooled-vs-single effect is large
LABEL_EFF_SEEDS = [42, 1, 7]                 # 3 seeds; 6 pairs x 7 budgets x 3 seeds
DANN_SEEDS      = [42, 1, 7]
K_GRID          = [0, 25, 50, 100, 250, 500, 1000]

# Training budget, matched to Notebook 18 so the tables merge.
MAX_LENGTH, EPOCHS, PATIENCE = 128, 8, 2
BATCH_SIZE, EVAL_BATCH_SIZE  = 32, 64
LEARNING_RATE, WEIGHT_DECAY, WARMUP_RATIO = 2e-5, 0.01, 0.10

# Few-shot adaptation budget (fixed; too few examples to justify a target validation split).
FEWSHOT_LR, FEWSHOT_BATCH = 2e-5, 16
def fewshot_epochs(k):
    return 8 if k <= 100 else (5 if k <= 500 else 3)

# ── Stage gates ───────────────────────────────────────────────────────────────
RUN_STAGE_A_10SEED    = True    # ~1.0 GPU-h
RUN_STAGE_B_LOCO      = True    # ~4.0 GPU-h
RUN_STAGE_C_LABELEFF  = True    # ~2.0 GPU-h
RUN_STAGE_D_DANN      = True    # ~1.5 GPU-h
RUN_STAGE_E_THRESHOLD = True    # CPU only
RUN_STAGE_F_NEARDUP   = True    # CPU only
RUN_STAGE_G_LLM       = False   # OFF by default: large download + long inference. See Stage G.

LLM_MODEL_NAME = os.getenv("NB19_LLM", "Qwen/Qwen2.5-7B-Instruct")
LLM_MAX_EVAL   = int(os.getenv("NB19_LLM_MAX_EVAL", "800"))   # per corpus, stratified subsample

OVERWRITE_COMPLETED = False
KEEP_TRAINER_CHECKPOINTS_AFTER_SUCCESS = False
BOOTSTRAP_B = 2000

# ── Cost guard ────────────────────────────────────────────────────────────────
HOURLY_RATE_USD  = float(os.getenv("RUNPOD_GPU_HOURLY_USD", "0.80"))
MAX_COMPUTE_USD  = float(os.getenv("NB19_MAX_COMPUTE_USD", "10.00")) 
EXPECTED_FIRST_JOB_SECONDS = float(os.getenv("NB19_FIRST_JOB_SECONDS", "300"))

# ── Table paths ───────────────────────────────────────────────────────────────
NB18_RUNS_CANDIDATES = [
    TABLES / "18_transformer_cross_corpus_multiseed_runs.csv",
    FT / "18_transformer_cross_corpus_multiseed_runs.csv",
]
A_RUNS   = TABLES / "19_singlesource_newseed_runs.csv"
A_MERGED = FT / "19_singlesource_10seed_runs.csv"
A_SUMM   = FT / "19_singlesource_10seed_summary.csv"
A_TESTS  = FT / "19_fgm_vs_vanilla_10seed_tests.csv"
B_RUNS   = TABLES / "19_loco_pooled_runs.csv"
B_SUMM   = FT / "19_loco_pooled_summary.csv"
B_COMP   = FT / "19_pooled_vs_single_source.csv"
C_RUNS   = TABLES / "19_label_efficiency_runs.csv"
C_SUMM   = FT / "19_label_efficiency_summary.csv"
C_KTO    = FT / "19_label_efficiency_k_to_reach.csv"
D_RUNS   = FT / "19_dann_loco_runs.csv"
E_TAB    = FT / "19_threshold_recalibration.csv"
F_AUDIT  = FT / "19_near_duplicate_audit.csv"
F_SENS   = FT / "19_near_duplicate_sensitivity.csv"
G_TAB    = FT / "19_llm_baselines_local.csv"
LEDGER   = FT / "19_claim_ledger.csv"
CONFIG_J = FT / "19_run_config.json"
MANIFEST = FT / "19_MANIFEST_sha256.json"

HAS_CUDA = torch.cuda.is_available()
HAS_BF16 = HAS_CUDA and torch.cuda.is_bf16_supported()
GPU_NAME = torch.cuda.get_device_name(0) if HAS_CUDA else "CPU"
NEEDS_GPU = any([RUN_STAGE_A_10SEED, RUN_STAGE_B_LOCO, RUN_STAGE_C_LABELEFF,
                 RUN_STAGE_D_DANN, RUN_STAGE_G_LLM])
if NEEDS_GPU and not HAS_CUDA:
    raise RuntimeError("A training stage is enabled but no CUDA GPU is visible. Use a RunPod GPU pod.")

print("ROOT        :", ROOT)
print("GPU         :", GPU_NAME, "| bf16:", HAS_BF16)
print("transformers:", transformers.__version__, "| torch:", torch.__version__)
print("budget cap  : $%.2f at $%.2f/hour" % (MAX_COMPUTE_USD, HOURLY_RATE_USD))

ROOT        : /workspace/Beyond_In_domain_Bangla_Sarcasm_Detection
GPU         : NVIDIA GeForce RTX 4090 | bf16: True
transformers: 4.57.6 | torch: 2.8.0+cu128
budget cap  : $10.00 at $0.80/hour


## Configuration freeze

The hash below is computed from the experimental configuration **before** any model is trained,
and is written together with the current git commit. Cite both in the manuscript's Limitations
section so a referee can verify that the confirmatory runs were not tuned after seeing results.

In [4]:
def git_commit():
    try:
        out = subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(ROOT),
                             capture_output=True, text=True, timeout=10)
        sha = out.stdout.strip()
        dirty = subprocess.run(["git", "status", "--porcelain"], cwd=str(ROOT),
                               capture_output=True, text=True, timeout=10).stdout.strip()
        return {"commit": sha or None, "dirty": bool(dirty)}
    except Exception:
        return {"commit": None, "dirty": None}

FROZEN_CONFIG = dict(
    notebook="19_ncaa_mitigation_and_power.ipynb",
    model_name=MODEL_NAME, corpora=CORPORA, systems=SYSTEMS,
    seeds_nb18=SEEDS_NB18, seeds_new=SEEDS_NEW, seeds_10=SEEDS_10,
    loco_seeds=LOCO_SEEDS, label_eff_seeds=LABEL_EFF_SEEDS, dann_seeds=DANN_SEEDS,
    k_grid=K_GRID, max_length=MAX_LENGTH, epochs=EPOCHS, patience=PATIENCE,
    batch_size=BATCH_SIZE, eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
    fewshot_lr=FEWSHOT_LR, fewshot_batch=FEWSHOT_BATCH,
    leakage_policy=("off-diagonal target test excludes normalized exact overlaps with "
                    "source train+val+test; pooled/LOCO sources use the union of all "
                    "constituent corpora for the same filter"),
    selection_policy="checkpoint and temperature selected on source validation only; test never used",
    fewshot_policy="k-shot examples sampled stratified from the target TRAIN split only",
    primary_uncertainty="Student-t 95% CI across training seeds",
    paired_test="exact sign-flip permutation over matched seed differences; Holm over 9 cells",
)
CONFIG_HASH = hashlib.sha256(
    json.dumps(FROZEN_CONFIG, sort_keys=True, default=str).encode("utf-8")
).hexdigest()[:16]

RUN_META = dict(
    config_hash=CONFIG_HASH, frozen_at=datetime.now(timezone.utc).isoformat(),
    git=git_commit(), gpu=GPU_NAME,
    versions=dict(python=platform.python_version(), torch=torch.__version__,
                  transformers=transformers.__version__, numpy=np.__version__,
                  pandas=pd.__version__, scipy=scipy.__version__),
    stages=dict(A=RUN_STAGE_A_10SEED, B=RUN_STAGE_B_LOCO, C=RUN_STAGE_C_LABELEFF,
                D=RUN_STAGE_D_DANN, E=RUN_STAGE_E_THRESHOLD, F=RUN_STAGE_F_NEARDUP,
                G=RUN_STAGE_G_LLM),
    config=FROZEN_CONFIG,
)
json.dump(RUN_META, open(CONFIG_J, "w"), indent=2, default=str)
print("CONFIG HASH :", CONFIG_HASH)
print("git commit  :", RUN_META["git"]["commit"], "| dirty:", RUN_META["git"]["dirty"])
print("frozen to   :", CONFIG_J.relative_to(ROOT))
if not RUN_META["git"]["commit"]:
    print("\\nWARNING: no git commit recorded. Commit the notebook before the confirmatory runs.")

CONFIG HASH : 6970de89ef36b0ad
git commit  : a64ee91ba5ee8b30b94f66437afb4228bce5414b | dirty: True
frozen to   : 04_outputs/finalized_outputs/tables/19_run_config.json


In [5]:
# Data contracts, normalization, and overlap control — byte-identical to Notebook 18
_ZW = dict.fromkeys(map(ord, ["\u200b", "\u200c", "\u200d", "\ufeff"]), None)

def norm_key(value):
    if not isinstance(value, str):
        value = "" if pd.isna(value) else str(value)
    value = unicodedata.normalize("NFC", value).translate(_ZW)
    return re.sub(r"\s+", " ", value).strip().casefold()

# Stage F only: stricter key that also collapses punctuation, digits and character elongation.
_PUNCT = re.compile(r"[^\w\s]", flags=re.UNICODE)
def near_key(value):
    v = norm_key(value)
    v = _PUNCT.sub("", v)
    v = re.sub(r"(.)\1{2,}", r"\1", v)      # elongation: "haaaaa" -> "ha"
    v = re.sub(r"\d+", "0", v)
    return re.sub(r"\s+", " ", v).strip()

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if HAS_CUDA:
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

def item_id(text, gold):
    return hashlib.sha256(f"{norm_key(text)}\n{int(gold)}".encode("utf-8")).hexdigest()[:20]

def read_split(corpus, split):
    path = SPLITS / f"{corpus}_{split}.csv"
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    need = {"text", "label_binary"}
    if not need.issubset(df.columns):
        raise ValueError(f"{path.name}: expected {sorted(need)}, got {list(df.columns)}")
    df = df.dropna(subset=["text", "label_binary"]).copy()
    df["text"] = df["text"].astype(str)
    df["label_binary"] = df["label_binary"].astype(int)
    if not set(df["label_binary"].unique()).issubset({0, 1}):
        raise ValueError(f"{path.name}: non-binary labels detected")
    df["norm_key_nb18"] = df["text"].map(norm_key)
    df["near_key"] = df["text"].map(near_key)
    df["item_id"] = [item_id(t, y) for t, y in zip(df.text, df.label_binary)]
    df["corpus"] = corpus
    if df["norm_key_nb18"].duplicated().any():
        raise ValueError(f"{path.name}: normalized duplicates remain inside split")
    return df.reset_index(drop=True)

DATA = {c: {s: read_split(c, s) for s in ("train", "val", "test")} for c in CORPORA}

for c in CORPORA:
    keys = {s: set(DATA[c][s]["norm_key_nb18"]) for s in ("train", "val", "test")}
    assert not (keys["train"] & keys["val"]) and not (keys["train"] & keys["test"]) \
        and not (keys["val"] & keys["test"]), f"{c}: internal split overlap"

# A "source" is now a tuple of corpora. Single-source keeps NB18 semantics exactly.
def source_label(sources):
    return "+".join(sources) if len(sources) > 1 else sources[0]

def source_frames(sources, seed=0):
    tr = pd.concat([DATA[c]["train"] for c in sources], ignore_index=True)
    va = pd.concat([DATA[c]["val"]   for c in sources], ignore_index=True)
    if len(sources) > 1:                       # deterministic shuffle for pooled training
        tr = tr.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        va = va.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return tr, va

def clean_target(sources, target):
    # Remove any target-test item whose normalized key appears anywhere in any source corpus.
    te = DATA[target]["test"].copy()
    if target in sources:
        return te, dict(n_eval_total=len(te), n_removed_train_seen=0,
                        n_removed_all_source=0, n_eval_clean=len(te))
    train_seen, all_source = set(), set()
    for s in sources:
        train_seen |= set(DATA[s]["train"]["norm_key_nb18"]) | set(DATA[s]["val"]["norm_key_nb18"])
        all_source |= train_seen | set(DATA[s]["test"]["norm_key_nb18"])
    k = te["norm_key_nb18"]
    removed_train = int(k.isin(train_seen).sum())
    keep = ~k.isin(all_source)
    clean = te.loc[keep].reset_index(drop=True)
    if set(clean["norm_key_nb18"]) & all_source:
        raise AssertionError(f"cross-corpus leakage remains: {sources} -> {target}")
    return clean, dict(n_eval_total=len(te), n_removed_train_seen=removed_train,
                       n_removed_all_source=int((~keep).sum()), n_eval_clean=len(clean))

LOCO_FOLDS = [(tuple(c for c in CORPORA if c != held), held) for held in CORPORA]
POOLED_ALL = tuple(CORPORA)

print("splits loaded:")
for c in CORPORA:
    print("  %-20s train=%6d val=%5d test=%5d  pos=%.3f"
          % (DISPLAY[c], len(DATA[c]["train"]), len(DATA[c]["val"]),
             len(DATA[c]["test"]), DATA[c]["test"].label_binary.mean()))
print("\\nLOCO folds:")
for src, held in LOCO_FOLDS:
    tr, va = source_frames(src)
    print("  hold out %-12s train on %-40s n_train=%6d" % (DISPLAY[held], source_label(list(src)), len(tr)))

splits loaded:
  Ben-Sarc             train= 20498 val= 2562 test= 2563  pos=0.500
  BanglaSarc           train=  3708 val=  463 test=  464  pos=0.358
  BanglaSarc3          train=  6328 val=  791 test=  791  pos=0.499
\nLOCO folds:
  hold out Ben-Sarc     train on banglasarc_binary+banglasarc3_binary     n_train= 10036
  hold out BanglaSarc   train on ben_sarc_binary+banglasarc3_binary       n_train= 26826
  hold out BanglaSarc3  train on ben_sarc_binary+banglasarc_binary        n_train= 24206


In [6]:
# Model, trainers, metrics, serialization — reused from Notebook 18 so results are comparable
class EncodedDataset(torch.utils.data.Dataset):
    def __init__(self, frame, tokenizer, max_length=128, with_domain=False, domain_map=None):
        self.frame = frame.reset_index(drop=True)
        self.enc = tokenizer(self.frame["text"].tolist(), truncation=True,
                             padding="max_length", max_length=max_length, return_tensors="pt")
        self.labels = torch.tensor(self.frame["label_binary"].values, dtype=torch.long)
        self.domains = None
        if with_domain:
            dm = domain_map or {}
            self.domains = torch.tensor([dm[c] for c in self.frame["corpus"]], dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        out = {k: v[idx] for k, v in self.enc.items()}
        out["labels"] = self.labels[idx]
        if self.domains is not None:
            out["domain"] = self.domains[idx]
        return out

class FGM:
    def __init__(self, model, epsilon=0.5, embedding_name="word_embeddings"):
        self.model, self.epsilon, self.embedding_name = model, epsilon, embedding_name
        self.backup = {}
    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.embedding_name in name and param.grad is not None:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if torch.isfinite(norm) and norm.item() > 0:
                    param.data.add_(self.epsilon * param.grad / norm)
    def restore(self):
        for name, value in self.backup.items():
            dict(self.model.named_parameters())[name].data.copy_(value)
        self.backup.clear()

class FGMTrainer(Trainer):
    def __init__(self, *args, fgm_epsilon=0.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.fgm = FGM(self.model, epsilon=fgm_epsilon)
    def training_step(self, model, inputs, num_items_in_batch=None):
        model.train()
        inputs = self._prepare_inputs(inputs)
        with self.compute_loss_context_manager():
            loss = self.compute_loss(model, inputs)
        if self.args.n_gpu > 1:
            loss = loss.mean()
        self.accelerator.backward(loss)
        self.fgm.attack()
        with self.compute_loss_context_manager():
            adversarial_loss = self.compute_loss(model, inputs)
        if self.args.n_gpu > 1:
            adversarial_loss = adversarial_loss.mean()
        self.accelerator.backward(adversarial_loss)
        self.fgm.restore()
        return loss.detach()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred = np.asarray(logits).argmax(axis=-1)
    return {"macro_f1": f1_score(labels, pred, average="macro", zero_division=0),
            "accuracy": accuracy_score(labels, pred)}

def softmax_np(logits):
    z = np.asarray(logits, dtype=np.float64)
    z -= z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def score(gold, pred):
    cr = recall_score(gold, pred, labels=[0, 1], average=None, zero_division=0)
    return dict(accuracy=float(accuracy_score(gold, pred)),
                macro_f1=float(f1_score(gold, pred, average="macro", zero_division=0)),
                weighted_f1=float(f1_score(gold, pred, average="weighted", zero_division=0)),
                precision_macro=float(precision_score(gold, pred, average="macro", zero_division=0)),
                recall_macro=float(recall_score(gold, pred, average="macro", zero_division=0)),
                recall_class_0=float(cr[0]), recall_class_1=float(cr[1]))

def expected_calibration_error(gold, probs, n_bins=15):
    gold = np.asarray(gold); probs = np.asarray(probs)
    conf = probs.max(axis=1); pred = probs.argmax(axis=1)
    edges = np.linspace(0.0, 1.0, n_bins + 1); ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (conf > lo) & (conf <= hi) if lo > 0 else (conf >= lo) & (conf <= hi)
        if mask.any():
            ece += mask.mean() * abs((pred[mask] == gold[mask]).mean() - conf[mask].mean())
    return float(ece)

def multiclass_brier(gold, probs):
    onehot = np.eye(probs.shape[1], dtype=float)[np.asarray(gold, dtype=int)]
    return float(np.mean(np.sum((np.asarray(probs) - onehot) ** 2, axis=1)))

def fit_temperature(logits, gold):
    logits = np.asarray(logits, dtype=np.float64); gold = np.asarray(gold, dtype=int)
    def objective(log_t):
        p = softmax_np(logits / np.exp(log_t))
        return float(-np.log(np.clip(p[np.arange(len(gold)), gold], 1e-12, 1.0)).mean())
    return float(np.exp(minimize_scalar(objective, bounds=(-2.3, 2.3), method="bounded").x))

def make_args(output_dir, seed, adversarial=False, epochs=None, lr=None,
              batch=None, eval_steps_strategy="epoch", patience_used=True):
    use_bf16 = bool(HAS_BF16)
    use_fp16 = bool(HAS_CUDA and not use_bf16 and not adversarial)
    common = dict(
        output_dir=str(output_dir), num_train_epochs=epochs or EPOCHS,
        per_device_train_batch_size=batch or BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=lr or LEARNING_RATE, weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO, lr_scheduler_type="linear",
        save_strategy=eval_steps_strategy if patience_used else "no",
        logging_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=bool(patience_used), metric_for_best_model="macro_f1",
        greater_is_better=True, report_to="none", seed=seed, data_seed=seed,
        bf16=use_bf16, fp16=use_fp16, dataloader_num_workers=2, remove_unused_columns=True,
    )
    try:
        return TrainingArguments(eval_strategy=eval_steps_strategy if patience_used else "no", **common)
    except TypeError:
        return TrainingArguments(evaluation_strategy=eval_steps_strategy if patience_used else "no", **common)

def save_prediction_file(path, frame, logits, system, source, target, seed, temperature, extra=None):
    probs = softmax_np(logits)
    calibrated = softmax_np(np.asarray(logits) / temperature)
    pred = probs.argmax(axis=1)
    out = pd.DataFrame({
        "item_id": frame["item_id"].values, "gold_label": frame["label_binary"].values,
        "pred_label": pred, "correct": (pred == frame["label_binary"].values).astype(int),
        "logit_0": np.asarray(logits)[:, 0], "logit_1": np.asarray(logits)[:, 1],
        "prob_0": probs[:, 0], "prob_1": probs[:, 1],
        "prob_cal_0": calibrated[:, 0], "prob_cal_1": calibrated[:, 1],
        "temperature": temperature, "system": system, "source": source,
        "target": target, "seed": seed,
    })
    for k, v in (extra or {}).items():
        out[k] = v
    path.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(path, index=False)
    return out

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print("tokenizer:", tokenizer.__class__.__name__)

tokenizer: ElectraTokenizerFast


In [7]:
# Resume bookkeeping and budget guard
NOTEBOOK_START = time.time()
_gpu_seconds_spent = [0.0]

def load_rows(path):
    return pd.read_csv(path).to_dict("records") if Path(path).exists() else []

def enforce_budget(pending_jobs, per_job_hint=None):
    per_job = per_job_hint or EXPECTED_FIRST_JOB_SECONDS
    spent_h = _gpu_seconds_spent[0] / 3600.0
    projected_h = spent_h + pending_jobs * per_job / 3600.0
    projected_cost = projected_h * HOURLY_RATE_USD
    print("  [budget] spent %.2f GPU-h ($%.2f) | projected %.2f GPU-h ($%.2f) | cap $%.2f"
          % (spent_h, spent_h * HOURLY_RATE_USD, projected_h, projected_cost, MAX_COMPUTE_USD))
    if projected_cost > MAX_COMPUTE_USD:
        raise RuntimeError(
            "Budget guard stopped before the next job: projected $%.2f > cap $%.2f. "
            "Disable a later stage or raise NB19_MAX_COMPUTE_USD after checking the pod rate."
            % (projected_cost, MAX_COMPUTE_USD))

def charge(seconds):
    _gpu_seconds_spent[0] += float(seconds)

def free_gpu():
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()

def train_source_model(sources, system, seed, tag, save_to=None, keep_trainer=False):
    # Train one model on the union of `sources`; returns (trainer, val_frame, val_output, seconds).
    set_seed(seed)
    src_label = source_label(list(sources))
    run_dir = CKPT / "_trainer_state" / f"{tag}_{system}_{src_label}_seed{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)
    tr, va = source_frames(list(sources), seed=seed)
    tr_ds, va_ds = EncodedDataset(tr, tokenizer, MAX_LENGTH), EncodedDataset(va, tokenizer, MAX_LENGTH)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    args = make_args(run_dir, seed, adversarial=(system == "fgm"))
    cls = FGMTrainer if system == "fgm" else Trainer
    kw = dict(model=model, args=args, train_dataset=tr_ds, eval_dataset=va_ds,
              compute_metrics=compute_metrics,
              callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    if system == "fgm":
        kw["fgm_epsilon"] = float(SYSTEMS[system]["fgm_epsilon"])
    trainer = cls(**kw)
    ckpts = sorted(run_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
    print("  [%s|%s|%s|seed %s] n_train=%d n_val=%d resume=%s"
          % (tag, system, src_label, seed, len(tr), len(va), bool(ckpts)))
    t0 = time.time()
    trainer.train(resume_from_checkpoint=str(ckpts[-1]) if ckpts else None)
    seconds = float(time.time() - t0)
    charge(seconds)
    val_out = trainer.predict(va_ds)
    if save_to is not None:
        Path(save_to).mkdir(parents=True, exist_ok=True)
        trainer.save_model(str(save_to)); tokenizer.save_pretrained(str(save_to))
    if not keep_trainer:
        shutil.rmtree(run_dir, ignore_errors=True)
    return trainer, va, val_out, seconds

def evaluate_on_targets(trainer, sources, system, seed, tag, temperature,
                        pred_subdir, targets=None, extra_row=None):
    rows = []
    for target in (targets or CORPORA):
        clean, audit = clean_target(list(sources), target)
        out = trainer.predict(EncodedDataset(clean, tokenizer, MAX_LENGTH))
        logits = np.asarray(out.predictions); pred = logits.argmax(axis=1)
        m = score(clean.label_binary.values, pred)
        probs, cal = softmax_np(logits), softmax_np(logits / temperature)
        src_label = source_label(list(sources))
        pth = PRED / pred_subdir / f"19_{tag}_{system}_{src_label}_seed{seed}_to_{target}.csv"
        save_prediction_file(pth, clean, logits, system, src_label, target, seed, temperature)
        rows.append(dict(
            tag=tag, system=system, source=src_label, source_corpora=json.dumps(list(sources)),
            target=target, seed=int(seed), held_out=bool(target not in sources),
            n_sources=len(sources), model_name=MODEL_NAME, **audit,
            test_macro_f1=m["macro_f1"], test_accuracy=m["accuracy"],
            test_weighted_f1=m["weighted_f1"], test_precision_macro=m["precision_macro"],
            test_recall_macro=m["recall_macro"], test_recall_class_0=m["recall_class_0"],
            test_recall_class_1=m["recall_class_1"], temperature=temperature,
            ece=expected_calibration_error(clean.label_binary.values, probs),
            ece_temperature_scaled=expected_calibration_error(clean.label_binary.values, cal),
            brier=multiclass_brier(clean.label_binary.values, probs),
            brier_temperature_scaled=multiclass_brier(clean.label_binary.values, cal),
            confusion_matrix=json.dumps(confusion_matrix(clean.label_binary.values, pred,
                                                         labels=[0, 1]).tolist()),
            prediction_path=str(pth.relative_to(ROOT)), config_hash=CONFIG_HASH,
            **(extra_row or {})))
        print("    -> %-12s F1=%.4f  (clean n=%d, removed=%d)"
              % (DISPLAY[target], m["macro_f1"], len(clean), audit["n_removed_all_source"]))
    return rows

print("helpers ready")

helpers ready


---
## Stage A — Single-source matrix at ten seeds

Notebook 18 completed seeds 42, 1, 7, 123, 2024. This stage trains only the five new seeds
(2025, 13, 77, 314, 1337) and merges both records into one 10-seed table.

**Why this matters more than anything else in the notebook.** With five seeds the exact two-sided
sign-flip test has a minimum attainable p of 2/2⁵ = 0.0625, so no result could ever reach 0.05 —
which is what the previous abstract was forced to admit. With ten seeds the minimum is
2/2¹⁰ = 0.00195, clearing even the Holm threshold for nine cells (0.0056).

Cost: 30 training jobs, roughly one GPU-hour.

In [8]:
nb18_runs = None
for cand in NB18_RUNS_CANDIDATES:
    if cand.exists():
        nb18_runs = pd.read_csv(cand)
        print("found Notebook 18 runs:", cand.relative_to(ROOT), "|", len(nb18_runs), "rows")
        break
if nb18_runs is None:
    print("WARNING: Notebook 18 run table not found. Stage A will train all 10 seeds.")
    nb18_runs = pd.DataFrame()

a_rows = load_rows(A_RUNS)
def a_done(rows):
    if not rows:
        return set()
    d = pd.DataFrame(rows); done = set()
    for (sysname, src, sd), g in d.groupby(["system", "source", "seed"]):
        if set(g["target"]) == set(CORPORA):
            done.add((str(sysname), str(src), int(sd)))
    return done

A_DONE = a_done(a_rows)
nb18_done = set()
if len(nb18_runs):
    for (sysname, src, sd), g in nb18_runs.groupby(["system", "source", "seed"]):
        if set(g["target"]) == set(CORPORA):
            nb18_done.add((str(sysname), str(src), int(sd)))

jobs_A = [(s, c, sd) for s in SYSTEMS for c in CORPORA for sd in SEEDS_10
          if (s, c, sd) not in nb18_done and (s, c, sd) not in A_DONE]
print("Stage A pending jobs:", len(jobs_A), "of", len(SYSTEMS) * len(CORPORA) * len(SEEDS_10))

if RUN_STAGE_A_10SEED and jobs_A:
    per_job = None
    for i, (system, source, seed) in enumerate(jobs_A):
        enforce_budget(len(jobs_A) - i, per_job)
        trainer, va, val_out, secs = train_source_model((source,), system, seed, "A")
        per_job = secs if per_job is None else 0.5 * (per_job + secs)
        val_pred = np.asarray(val_out.predictions).argmax(axis=1)
        vm = score(va.label_binary.values, val_pred)
        temp = fit_temperature(val_out.predictions, va.label_binary.values)
        np.save(CKPT / f"valLogits_A_{system}_{source}_seed{seed}.npy",
                np.asarray(val_out.predictions))
        pd.DataFrame({"gold": va.label_binary.values}).to_csv(
            CKPT / f"valGold_A_{system}_{source}_seed{seed}.csv", index=False)
        rows = evaluate_on_targets(trainer, (source,), system, seed, "A", temp,
                                   "19_singlesource_newseeds",
                                   extra_row=dict(val_macro_f1=vm["macro_f1"],
                                                  val_accuracy=vm["accuracy"],
                                                  train_seconds=secs))
        a_rows = [r for r in a_rows
                  if (str(r.get("system")), str(r.get("source")), int(r.get("seed", -1)))
                  != (system, source, int(seed))]
        a_rows.extend(rows)
        pd.DataFrame(a_rows).to_csv(A_RUNS, index=False)
        del trainer; free_gpu()
        print("  saved %s (%d rows)" % (A_RUNS.name, len(a_rows)))
elif not RUN_STAGE_A_10SEED:
    print("Stage A disabled.")
else:
    print("Stage A already complete.")

found Notebook 18 runs: 04_outputs/tables/18_transformer_cross_corpus_multiseed_runs.csv | 90 rows
Stage A pending jobs: 0 of 60
Stage A already complete.


In [9]:
# Merge Notebook 18 and Stage A into one 10-seed record
frames = []
if len(nb18_runs):
    n18 = nb18_runs.copy(); n18["tag"] = "NB18"; n18["n_sources"] = 1
    n18["source_corpora"] = n18["source"].map(lambda s: json.dumps([s]))
    n18["held_out"] = ~n18["in_domain"].astype(bool)
    frames.append(n18)
if Path(A_RUNS).exists():
    frames.append(pd.read_csv(A_RUNS))
merged = pd.concat(frames, ignore_index=True, sort=False)
merged["in_domain"] = merged["source"] == merged["target"]
merged = merged.drop_duplicates(["system", "source", "target", "seed"], keep="last")
merged.to_csv(A_MERGED, index=False)

n_seeds_actual = merged.groupby(["system", "source", "target"])["seed"].nunique()
print("10-seed table:", len(merged), "rows | seeds per cell:",
      sorted(n_seeds_actual.unique()))

def tci(values):
    v = np.asarray(values, dtype=float); n = len(v)
    m, sd = float(v.mean()), float(v.std(ddof=1)) if n > 1 else 0.0
    if n < 2:
        return m, 0.0, m, m
    h = stats.t.ppf(0.975, n - 1) * sd / math.sqrt(n)
    return m, sd, m - h, m + h

summ = []
for (sysname, src, tgt), g in merged.groupby(["system", "source", "target"]):
    m, sd, lo, hi = tci(g.test_macro_f1)
    am, asd, alo, ahi = tci(g.test_accuracy)
    summ.append(dict(system=sysname, source=src, target=tgt,
                     in_domain=bool(src == tgt), seeds=int(g.seed.nunique()),
                     macro_f1_mean=m, macro_f1_std=sd, macro_f1_ci95_lo=lo, macro_f1_ci95_hi=hi,
                     accuracy_mean=am, accuracy_std=asd,
                     ece_mean=float(g.ece.mean()),
                     ece_temperature_scaled_mean=float(g.ece_temperature_scaled.mean()),
                     brier_mean=float(g.brier.mean()),
                     recall_class_0_mean=float(g.test_recall_class_0.mean()),
                     recall_class_1_mean=float(g.test_recall_class_1.mean()),
                     n_eval_clean=int(g.n_eval_clean.iloc[0])))
summary10 = pd.DataFrame(summ).sort_values(["system", "source", "target"]).reset_index(drop=True)
summary10.to_csv(A_SUMM, index=False)

def ceiling_for(system, corpus):
    # Target-trained diagonal mean for a system/corpus; None when unavailable.
    d = summary10[(summary10.system == system) & (summary10.source == corpus) &
                  (summary10.target == corpus)]
    return float(d.macro_f1_mean.iloc[0]) if len(d) else None

for sysname in ("fgm", "vanilla"):
    d = summary10[summary10.system == sysname]
    print("%-8s in-domain mean %.4f | off-diagonal mean %.4f"
          % (sysname, d[d.in_domain].macro_f1_mean.mean(), d[~d.in_domain].macro_f1_mean.mean()))
display(summary10[summary10.system == "fgm"][
    ["source", "target", "seeds", "macro_f1_mean", "macro_f1_std",
     "macro_f1_ci95_lo", "macro_f1_ci95_hi"]])

10-seed table: 180 rows | seeds per cell: [10]
fgm      in-domain mean 0.8472 | off-diagonal mean 0.5342
vanilla  in-domain mean 0.8417 | off-diagonal mean 0.5357


,source,target,seeds,macro_f1_mean,macro_f1_std,macro_f1_ci95_lo,macro_f1_ci95_hi
0,banglasarc3_binary,banglasarc3_binary,10,0.761723,0.008525,0.755625,0.767821
1,banglasarc3_binary,banglasarc_binary,10,0.611250,0.039237,0.583181,0.639319
2,banglasarc3_binary,ben_sarc_binary,10,0.666989,0.008924,0.660605,0.673372
3,banglasarc_binary,banglasarc3_binary,10,0.335098,0.004719,0.331722,0.338474
4,banglasarc_binary,banglasarc_binary,10,0.980235,0.004415,0.977077,0.983393
5,banglasarc_binary,ben_sarc_binary,10,0.345826,0.005087,0.342187,0.349465
6,ben_sarc_binary,banglasarc3_binary,10,0.651588,0.011171,0.643597,0.659579
7,ben_sarc_binary,banglasarc_binary,10,0.594287,0.032983,0.570693,0.617882
8,ben_sarc_binary,ben_sarc_binary,10,0.799750,0.004756,0.796348,0.803152


In [10]:
# Powered exact matched-seed FGM-vs-vanilla tests
def exact_sign_flip_test(differences):
    d = np.asarray(differences, dtype=float)
    if len(d) > 22:
        rng = np.random.default_rng(0)
        signs = rng.choice([-1.0, 1.0], size=(200000, len(d)))
        null = np.abs((d * signs).mean(axis=1))
        return float(np.mean(null >= abs(d.mean())))
    observed, vals = abs(d.mean()), []
    for bits in range(2 ** len(d)):
        s = np.array([1 if (bits >> i) & 1 else -1 for i in range(len(d))])
        vals.append(abs((d * s).mean()))
    return float(np.mean(np.asarray(vals) >= observed))

def holm(pvals):
    p = np.asarray(pvals, dtype=float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * p[idx])
        adj[idx] = min(1.0, running)
    return adj

test_rows = []
for src in CORPORA:
    for tgt in CORPORA:
        piv = (merged[merged.source.eq(src) & merged.target.eq(tgt)]
               .pivot_table(index="seed", columns="system", values="test_macro_f1"))
        piv = piv.dropna()
        if not {"fgm", "vanilla"}.issubset(piv.columns) or len(piv) < 2:
            continue
        d = (piv["fgm"] - piv["vanilla"]).values
        m, sd, lo, hi = tci(d)
        test_rows.append(dict(source=src, target=tgt, in_domain=bool(src == tgt),
                              n_seeds=len(d), vanilla_mean=float(piv["vanilla"].mean()),
                              fgm_mean=float(piv["fgm"].mean()), delta_mean=m, delta_std=sd,
                              delta_ci95_lo=lo, delta_ci95_hi=hi,
                              exact_sign_flip_p=exact_sign_flip_test(d)))
tests10 = pd.DataFrame(test_rows)
if len(tests10):
    tests10["p_holm_9_cells"] = holm(tests10.exact_sign_flip_p.values)
    tests10["significant_holm_0_05"] = tests10.p_holm_9_cells < 0.05
    tests10.to_csv(A_TESTS, index=False)
    n = int(tests10.n_seeds.min())
    print("minimum attainable two-sided p with %d matched seeds: %.5f" % (n, 2 / 2 ** n))
    print("Holm threshold for the smallest of 9 hypotheses: %.5f" % (0.05 / 9))
    display(tests10[["source", "target", "n_seeds", "vanilla_mean", "fgm_mean", "delta_mean",
                     "exact_sign_flip_p", "p_holm_9_cells", "significant_holm_0_05"]])

minimum attainable two-sided p with 10 matched seeds: 0.00195
Holm threshold for the smallest of 9 hypotheses: 0.00556


,source,target,n_seeds,vanilla_mean,fgm_mean,delta_mean,exact_sign_flip_p,p_holm_9_cells,significant_holm_0_05
0,ben_sarc_binary,ben_sarc_binary,10,0.793721,0.799750,0.006029,0.007812,0.070312,False
1,ben_sarc_binary,banglasarc_binary,10,0.585481,0.594287,0.008806,0.236328,1.000000,False
2,ben_sarc_binary,banglasarc3_binary,10,0.644825,0.651588,0.006763,0.029297,0.234375,False
3,banglasarc_binary,ben_sarc_binary,10,0.346680,0.345826,-0.000854,0.482422,1.000000,False
4,banglasarc_binary,banglasarc_binary,10,0.974905,0.980235,0.005330,0.046875,0.328125,False
5,banglasarc_binary,banglasarc3_binary,10,0.336533,0.335098,-0.001435,0.355469,1.000000,False
6,banglasarc3_binary,ben_sarc_binary,10,0.672796,0.666989,-0.005808,0.193359,1.000000,False
7,banglasarc3_binary,banglasarc_binary,10,0.627863,0.611250,-0.016613,0.304688,1.000000,False
8,banglasarc3_binary,banglasarc3_binary,10,0.756590,0.761723,0.005133,0.279297,1.000000,False


---
## Stage B — Leave-one-corpus-out and pooled multi-source training

Three LOCO folds (train on two corpora, evaluate on the fully held-out third) plus one
pooled-all configuration, for both systems, across five seeds.

LOCO is the deployment-realistic protocol and should become the manuscript's headline table:
it answers the question a practitioner actually has — *"I have two Bengali sarcasm datasets and a
new platform; what do I get?"* No prior Bengali sarcasm paper reports it.

Cost: 40 training jobs on larger training sets, roughly four GPU-hours.

In [11]:
b_rows = load_rows(B_RUNS)
def b_done(rows):
    if not rows:
        return set()
    d = pd.DataFrame(rows); done = set()
    for (sysname, src, sd), g in d.groupby(["system", "source", "seed"]):
        done.add((str(sysname), str(src), int(sd)))
    return done
B_DONE = b_done(b_rows)

configs_B = [(tuple(src), held) for src, held in LOCO_FOLDS] + [(POOLED_ALL, None)]
jobs_B = [(sysname, src, held, sd)
          for sysname in SYSTEMS for src, held in configs_B for sd in LOCO_SEEDS
          if (sysname, source_label(list(src)), sd) not in B_DONE]
print("Stage B pending jobs:", len(jobs_B), "of", len(SYSTEMS) * len(configs_B) * len(LOCO_SEEDS))

if RUN_STAGE_B_LOCO and jobs_B:
    per_job = None
    for i, (system, sources, held, seed) in enumerate(jobs_B):
        enforce_budget(len(jobs_B) - i, per_job)
        trainer, va, val_out, secs = train_source_model(sources, system, seed, "B")
        per_job = secs if per_job is None else 0.5 * (per_job + secs)
        vm = score(va.label_binary.values, np.asarray(val_out.predictions).argmax(axis=1))
        temp = fit_temperature(val_out.predictions, va.label_binary.values)
        rows = evaluate_on_targets(
            trainer, sources, system, seed, "B", temp, "19_loco_pooled",
            extra_row=dict(protocol=("loco" if held else "pooled_all"),
                           held_out_corpus=held or "", val_macro_f1=vm["macro_f1"],
                           val_accuracy=vm["accuracy"], train_seconds=secs))
        key = (system, source_label(list(sources)), int(seed))
        b_rows = [r for r in b_rows
                  if (str(r.get("system")), str(r.get("source")), int(r.get("seed", -1))) != key]
        b_rows.extend(rows)
        pd.DataFrame(b_rows).to_csv(B_RUNS, index=False)
        del trainer; free_gpu()
        print("  saved %s (%d rows)" % (B_RUNS.name, len(b_rows)))
elif not RUN_STAGE_B_LOCO:
    print("Stage B disabled.")
else:
    print("Stage B already complete.")

Stage B pending jobs: 17 of 40
  [budget] spent 0.00 GPU-h ($0.00) | projected 1.42 GPU-h ($1.13) | cap $10.00


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  [B|fgm|banglasarc_binary+banglasarc3_binary|seed 123] n_train=10036 n_val=1254 resume=True


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
3,0.235300,0.415901,0.822162,0.823764
4,0.121800,0.501470,0.816971,0.818182
5,0.062000,0.587034,0.801054,0.803030


    -> Ben-Sarc     F1=0.4961  (clean n=2563, removed=0)


...

  [B|fgm|ben_sarc_binary+banglasarc_binary+banglasarc3_binary|seed 2024] n_train=30534 n_val=3816 resume=False


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.566400,0.482174,0.774061,0.774633
2,0.418500,0.450594,0.793504,0.794287
3,0.270800,0.519495,0.787474,0.788260
4,0.149600,0.609863,0.776867,0.778302


    -> Ben-Sarc     F1=0.7904  (clean n=2563, removed=0)


    -> BanglaSarc   F1=0.8654  (clean n=464, removed=0)


    -> BanglaSarc3  F1=0.7392  (clean n=791, removed=0)
  saved 19_loco_pooled_runs.csv (120 rows)


In [12]:
# Stage B aggregation: LOCO held-out performance vs single-source transfer vs target-trained ceiling
loco_summary = pd.DataFrame()
if Path(B_RUNS).exists():
    bdf = pd.read_csv(B_RUNS)
    rows = []
    for (sysname, src, tgt, proto), g in bdf.groupby(["system", "source", "target", "protocol"]):
        m, sd, lo, hi = tci(g.test_macro_f1)
        rows.append(dict(system=sysname, protocol=proto, source=src, target=tgt,
                         held_out=bool(g.held_out.iloc[0]), seeds=int(g.seed.nunique()),
                         macro_f1_mean=m, macro_f1_std=sd,
                         macro_f1_ci95_lo=lo, macro_f1_ci95_hi=hi,
                         accuracy_mean=float(g.test_accuracy.mean()),
                         recall_class_1_mean=float(g.test_recall_class_1.mean()),
                         ece_mean=float(g.ece.mean()),
                         ece_temperature_scaled_mean=float(g.ece_temperature_scaled.mean()),
                         n_eval_clean=int(g.n_eval_clean.iloc[0])))
    loco_summary = pd.DataFrame(rows).sort_values(["system", "protocol", "target"]).reset_index(drop=True)
    loco_summary.to_csv(B_SUMM, index=False)

    # Headline comparison: for each target, LOCO vs the best single-source transfer vs the ceiling
    comp = []
    for sysname in SYSTEMS:
        s10 = summary10[summary10.system == sysname]
        for target in CORPORA:
            ceiling = ceiling_for(sysname, target)
            if ceiling is None:
                continue
            singles = s10[(s10.target == target) & (s10.source != target)]
            best_single = float(singles.macro_f1_mean.max())
            mean_single = float(singles.macro_f1_mean.mean())
            loco = loco_summary[(loco_summary.system == sysname) &
                                (loco_summary.protocol == "loco") &
                                (loco_summary.target == target) & loco_summary.held_out]
            if not len(loco):
                continue
            loco_f1 = float(loco.macro_f1_mean.iloc[0])
            comp.append(dict(
                system=sysname, target=target, target_trained_ceiling=ceiling,
                mean_single_source_transfer=mean_single, best_single_source_transfer=best_single,
                loco_two_source=loco_f1,
                gain_vs_mean_single=loco_f1 - mean_single,
                gain_vs_best_single=loco_f1 - best_single,
                retention_single_mean=mean_single / ceiling,
                retention_loco=loco_f1 / ceiling,
                gap_closed_fraction=((loco_f1 - mean_single) / (ceiling - mean_single)
                                     if ceiling > mean_single else np.nan),
                loco_ci95_lo=float(loco.macro_f1_ci95_lo.iloc[0]),
                loco_ci95_hi=float(loco.macro_f1_ci95_hi.iloc[0])))
    pooled_vs_single = pd.DataFrame(comp)
    pooled_vs_single.to_csv(B_COMP, index=False)
    display(pooled_vs_single.round(4))
    fg = pooled_vs_single[pooled_vs_single.system == "fgm"]
    if len(fg):
        print("\\nFGM: mean LOCO gain over mean single-source transfer = %+.4f macro-F1"
              % fg.gain_vs_mean_single.mean())
        print("FGM: mean retention rises from %.3f (single source) to %.3f (LOCO)"
              % (fg.retention_single_mean.mean(), fg.retention_loco.mean()))
else:
    pooled_vs_single = pd.DataFrame()
    print("Stage B produced no runs.")

,system,target,target_trained_ceiling,mean_single_source_transfer,best_single_source_transfer,loco_two_source,gain_vs_mean_single,gain_vs_best_single,retention_single_mean,retention_loco,gap_closed_fraction,loco_ci95_lo,loco_ci95_hi
0,vanilla,ben_sarc_binary,0.7937,0.5097,0.6728,0.5018,-0.0080,-0.1710,0.6422,0.6322,-0.0280,0.4804,0.5232
1,vanilla,banglasarc_binary,0.9749,0.6067,0.6279,0.5942,-0.0125,-0.0336,0.6223,0.6095,-0.0338,0.5600,0.6285
2,vanilla,banglasarc3_binary,0.7566,0.4907,0.6448,0.6470,0.1563,0.0022,0.6485,0.8552,0.5880,0.6317,0.6624
3,fgm,ben_sarc_binary,0.7998,0.5064,0.6670,0.5087,0.0023,-0.1583,0.6332,0.6360,0.0077,0.4969,0.5205
4,fgm,banglasarc_binary,0.9802,0.6028,0.6113,0.6207,0.0180,0.0095,0.6149,0.6332,0.0476,0.5832,0.6583
5,fgm,banglasarc3_binary,0.7617,0.4933,0.6516,0.6386,0.1453,-0.0129,0.6477,0.8384,0.5414,0.6171,0.6602


\nFGM: mean LOCO gain over mean single-source transfer = +0.0552 macro-F1
FGM: mean retention rises from 0.632 (single source) to 0.703 (LOCO)


---
## Stage C — Target label-efficiency curves

**This is the paper's practical payload.** For each directed source→target pair we fine-tune the
source model on k ∈ {0, 25, 50, 100, 250, 500, 1000} labelled target examples and measure how much
of the target-trained ceiling is recovered.

The sentence this earns: *"Fewer than N labelled target examples recover 90% of target-trained
macro-F1, so cross-corpus deployment is a small-annotation problem rather than an architecture
problem."* That is what turns a diagnosis into something an NCA reader can act on.

Protocol notes: the k examples are drawn stratified from the target **train** split only, using the
run seed. Adaptation uses a fixed epoch budget rather than early stopping, because a target
validation split is not meaningful at k = 25. The evaluation population is the same overlap-filtered
target test set used everywhere else.

Cost: 9 base models plus 108 short adaptations, roughly two GPU-hours.

In [13]:
c_rows = load_rows(C_RUNS)
C_DONE = {(str(r["source"]), str(r["target"]), int(r["seed"]), int(r["k"])) for r in c_rows} if c_rows else set()
LABEL_EFF_SYSTEM = "fgm"     # the primary system; keeps this stage affordable

pairs = [(s, t) for s in CORPORA for t in CORPORA if s != t]
pending_C = sum(1 for s, t in pairs for sd in LABEL_EFF_SEEDS for k in K_GRID
                if (s, t, sd, k) not in C_DONE)
print("Stage C pending evaluations:", pending_C, "of", len(pairs) * len(LABEL_EFF_SEEDS) * len(K_GRID))

def sample_kshot(target, k, seed):
    tr = DATA[target]["train"]
    if k >= len(tr):
        return tr.copy()
    rng = np.random.default_rng(seed)
    picks = []
    for label, g in tr.groupby("label_binary"):
        share = max(1, int(round(k * len(g) / len(tr))))
        idx = rng.choice(g.index.values, size=min(share, len(g)), replace=False)
        picks.append(tr.loc[idx])
    out = pd.concat(picks).drop_duplicates("item_id")
    if len(out) > k:
        out = out.sample(n=k, random_state=seed)
    return out.reset_index(drop=True)

def adapt_and_score(base_dir, target, k, seed, clean, audit):
    # k=0 evaluates the base model unchanged; k>0 fine-tunes a fresh copy on k target examples.
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(str(base_dir), num_labels=2)
    seconds = 0.0
    if k > 0:
        shot = sample_kshot(target, k, seed)
        run_dir = CKPT / "_fewshot" / f"{target}_k{k}_seed{seed}"
        run_dir.mkdir(parents=True, exist_ok=True)
        args = make_args(run_dir, seed, adversarial=False, epochs=fewshot_epochs(k),
                         lr=FEWSHOT_LR, batch=FEWSHOT_BATCH, patience_used=False)
        tr = Trainer(model=model, args=args,
                     train_dataset=EncodedDataset(shot, tokenizer, MAX_LENGTH),
                     compute_metrics=compute_metrics)
        t0 = time.time(); tr.train(); seconds = float(time.time() - t0); charge(seconds)
        shutil.rmtree(run_dir, ignore_errors=True)
        n_shot = len(shot)
    else:
        tr = Trainer(model=model, args=make_args(CKPT / "_eval_only", seed, patience_used=False),
                     compute_metrics=compute_metrics)
        n_shot = 0
    out = tr.predict(EncodedDataset(clean, tokenizer, MAX_LENGTH))
    logits = np.asarray(out.predictions); pred = logits.argmax(axis=1)
    m = score(clean.label_binary.values, pred)
    del tr, model; free_gpu()
    return m, n_shot, seconds

if RUN_STAGE_C_LABELEFF and pending_C:
    per_job = None
    for source in CORPORA:
        for seed in LABEL_EFF_SEEDS:
            needed = [(t, k) for t in CORPORA if t != source
                      for k in K_GRID if (source, t, seed, k) not in C_DONE]
            if not needed:
                continue
            base_dir = CKPT / "_base" / f"{LABEL_EFF_SYSTEM}_{source}_seed{seed}"
            if not (base_dir / "config.json").exists():
                enforce_budget(len(needed) + 1, per_job)
                trainer, va, val_out, secs = train_source_model(
                    (source,), LABEL_EFF_SYSTEM, seed, "C", save_to=base_dir)
                per_job = secs if per_job is None else 0.5 * (per_job + secs)
                del trainer; free_gpu()
            for target, k in needed:
                clean, audit = clean_target([source], target)
                m, n_shot, secs = adapt_and_score(base_dir, target, k, seed, clean, audit)
                c_rows = [r for r in c_rows
                          if (str(r.get("source")), str(r.get("target")),
                              int(r.get("seed", -1)), int(r.get("k", -1)))
                          != (source, target, int(seed), int(k))]
                c_rows.append(dict(
                    system=LABEL_EFF_SYSTEM, source=source, target=target, seed=int(seed),
                    k=int(k), n_shot_actual=int(n_shot), macro_f1=m["macro_f1"],
                    accuracy=m["accuracy"], recall_class_0=m["recall_class_0"],
                    recall_class_1=m["recall_class_1"], n_eval_clean=audit["n_eval_clean"],
                    adapt_seconds=secs, config_hash=CONFIG_HASH))
                pd.DataFrame(c_rows).to_csv(C_RUNS, index=False)
                print("  %-12s -> %-12s seed %-5s k=%-5d F1=%.4f"
                      % (DISPLAY[source], DISPLAY[target], seed, k, m["macro_f1"]))
            shutil.rmtree(base_dir, ignore_errors=True)
elif not RUN_STAGE_C_LABELEFF:
    print("Stage C disabled.")
else:
    print("Stage C already complete.")

Stage C pending evaluations: 126 of 126
  [budget] spent 1.96 GPU-h ($1.57) | projected 3.21 GPU-h ($2.57) | cap $10.00


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  [C|fgm|ben_sarc_binary|seed 42] n_train=20498 n_val=2562 resume=False


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.558000,0.448815,0.791563,0.791569
2,0.404200,0.444964,0.796718,0.797424
3,0.264100,0.475478,0.799724,0.799766
4,0.140700,0.581774,0.799728,0.799766
5,0.074300,0.681990,0.789960,0.790398
6,0.043100,0.838770,0.776560,0.777908


  Ben-Sarc     -> BanglaSarc   seed 42    k=0     F1=0.6510


...

  BanglaSarc3  -> BanglaSarc   seed 7     k=250   F1=0.9387


Step,Training Loss
32,0.490600
64,0.137200
96,0.033600
128,0.010400
160,0.007400


  BanglaSarc3  -> BanglaSarc   seed 7     k=500   F1=0.9415


Step,Training Loss
63,0.367400
126,0.065300
189,0.018800


  BanglaSarc3  -> BanglaSarc   seed 7     k=1000  F1=0.9634


In [14]:
# Stage C aggregation: recovery curves and the k needed to reach 80/90/95% of the ceiling
labeleff_summary, k_to_reach = pd.DataFrame(), pd.DataFrame()
if Path(C_RUNS).exists():
    cdf = pd.read_csv(C_RUNS)
    ceil_map = {t: ceiling_for(LABEL_EFF_SYSTEM, t) for t in CORPORA}
    if any(v is None for v in ceil_map.values()):
        raise RuntimeError("Stage C needs the Stage A diagonal means; run Stage A first.")
    rows = []
    for (src, tgt, k), g in cdf.groupby(["source", "target", "k"]):
        m, sd, lo, hi = tci(g.macro_f1)
        ceiling = ceil_map[tgt]
        rows.append(dict(system=LABEL_EFF_SYSTEM, source=src, target=tgt, k=int(k),
                         seeds=int(g.seed.nunique()), macro_f1_mean=m, macro_f1_std=sd,
                         macro_f1_ci95_lo=lo, macro_f1_ci95_hi=hi,
                         target_trained_ceiling=ceiling, retention=m / ceiling,
                         recall_class_1_mean=float(g.recall_class_1.mean())))
    labeleff_summary = pd.DataFrame(rows).sort_values(["source", "target", "k"]).reset_index(drop=True)
    labeleff_summary.to_csv(C_SUMM, index=False)

    reach = []
    for (src, tgt), g in labeleff_summary.groupby(["source", "target"]):
        g = g.sort_values("k")
        row = dict(source=src, target=tgt, zero_shot=float(g[g.k == 0].macro_f1_mean.iloc[0]),
                   ceiling=float(g.target_trained_ceiling.iloc[0]),
                   best_k=int(g.k.max()), best_f1=float(g.macro_f1_mean.max()))
        for thr in (0.80, 0.90, 0.95):
            hit = g[g.retention >= thr]
            row["k_for_%d_pct" % int(thr * 100)] = int(hit.k.iloc[0]) if len(hit) else np.nan
        reach.append(row)
    k_to_reach = pd.DataFrame(reach)
    k_to_reach.to_csv(C_KTO, index=False)
    display(k_to_reach.round(4))
    med90 = k_to_reach["k_for_90_pct"].dropna()
    if len(med90):
        print("\\nHEADLINE: %d of %d directed pairs reach 90%% of target-trained macro-F1 "
              "within the tested budget; median k = %d labelled target examples."
              % (len(med90), len(k_to_reach), int(med90.median())))
else:
    print("Stage C produced no runs.")

,source,target,zero_shot,ceiling,best_k,best_f1,k_for_80_pct,k_for_90_pct,k_for_95_pct
0,banglasarc3_binary,banglasarc_binary,0.6396,0.9802,1000,0.9587,50,100.0,500.0
1,banglasarc3_binary,ben_sarc_binary,0.6734,0.7998,1000,0.7565,0,50.0,NaN
2,banglasarc_binary,banglasarc3_binary,0.3350,0.7617,1000,0.7228,100,500.0,NaN
3,banglasarc_binary,ben_sarc_binary,0.3476,0.7998,1000,0.7147,250,NaN,NaN
4,ben_sarc_binary,banglasarc3_binary,0.6526,0.7617,1000,0.7454,0,25.0,500.0
5,ben_sarc_binary,banglasarc_binary,0.6102,0.9802,1000,0.9471,100,250.0,1000.0


\nHEADLINE: 5 of 6 directed pairs reach 90% of target-trained macro-F1 within the tested budget; median k = 100 labelled target examples.


---
## Stage D — DANN (gradient reversal) on the LOCO folds

One standard domain-generalization comparator, so the manuscript can say it tested a recognised
method rather than only measuring the problem. A domain classifier predicts which source corpus an
example came from; a gradient reversal layer pushes the encoder toward features that are useful for
sarcasm but uninformative about corpus identity. The adversarial weight follows the usual
Ganin–Lempitsky ramp.

If DANN does not beat plain pooled training, that is a perfectly publishable result for an
evaluation-focused paper — and a more honest one than a tuned-until-it-wins comparison. This stage
uses a plain PyTorch loop rather than `Trainer`, and is wrapped so a failure cannot lose Stages A–C.

Cost: 9 training jobs on LOCO-sized data, roughly 1.5 GPU-hours.

In [15]:
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambd, None

class DANNModel(nn.Module):
    def __init__(self, model_name, n_labels=2, n_domains=2, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, n_labels)
        self.domain_head = nn.Sequential(
            nn.Linear(hidden, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, n_domains))
    def forward(self, input_ids, attention_mask, token_type_ids=None, lambd=0.0):
        kw = dict(input_ids=input_ids, attention_mask=attention_mask)
        if token_type_ids is not None:
            kw["token_type_ids"] = token_type_ids
        h = self.encoder(**kw).last_hidden_state[:, 0]     # [CLS]
        h = self.dropout(h)
        return self.classifier(h), self.domain_head(GradReverse.apply(h, lambd))

@torch.no_grad()
def dann_predict(model, frame, device, batch=EVAL_BATCH_SIZE):
    model.eval()
    ds = EncodedDataset(frame, tokenizer, MAX_LENGTH)
    dl = torch.utils.data.DataLoader(ds, batch_size=batch, shuffle=False)
    outs = []
    for b in dl:
        b = {k: v.to(device) for k, v in b.items() if k != "labels"}
        logits, _ = model(**b, lambd=0.0)
        outs.append(logits.float().cpu().numpy())
    return np.concatenate(outs, axis=0)

def run_dann(sources, held, seed, epochs=4, lr=2e-5, batch=32, dann_alpha=1.0):
    set_seed(seed)
    device = torch.device("cuda" if HAS_CUDA else "cpu")
    dmap = {c: i for i, c in enumerate(sources)}
    tr, va = source_frames(list(sources), seed=seed)
    ds = EncodedDataset(tr, tokenizer, MAX_LENGTH, with_domain=True, domain_map=dmap)
    dl = torch.utils.data.DataLoader(ds, batch_size=batch, shuffle=True, drop_last=True)
    model = DANNModel(MODEL_NAME, n_domains=len(sources)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    total = max(1, epochs * len(dl))
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, total_steps=total, pct_start=WARMUP_RATIO, anneal_strategy="linear")
    ce = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler(enabled=bool(HAS_CUDA and not HAS_BF16))
    step, t0 = 0, time.time()
    for ep in range(epochs):
        model.train()
        for b in dl:
            p = step / total
            lambd = dann_alpha * (2.0 / (1.0 + math.exp(-10 * p)) - 1.0)
            labels = b.pop("labels").to(device); domain = b.pop("domain").to(device)
            b = {k: v.to(device) for k, v in b.items()}
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.bfloat16 if HAS_BF16 else torch.float16,
                                enabled=bool(HAS_CUDA)):
                logits, dlogits = model(**b, lambd=lambd)
                loss = ce(logits, labels) + ce(dlogits, domain)
            if scaler.is_enabled():
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss.backward(); opt.step()
            sched.step(); step += 1
        vp = dann_predict(model, va, device).argmax(axis=1)
        print("    epoch %d val macro-F1 %.4f" % (ep + 1, f1_score(va.label_binary.values, vp,
                                                                   average="macro", zero_division=0)))
    seconds = float(time.time() - t0); charge(seconds)
    val_logits = dann_predict(model, va, device)
    temp = fit_temperature(val_logits, va.label_binary.values)
    rows = []
    for target in CORPORA:
        clean, audit = clean_target(list(sources), target)
        logits = dann_predict(model, clean, device); pred = logits.argmax(axis=1)
        m = score(clean.label_binary.values, pred)
        pth = PRED / "19_dann" / f"19_D_dann_{source_label(list(sources))}_seed{seed}_to_{target}.csv"
        save_prediction_file(pth, clean, logits, "dann", source_label(list(sources)),
                             target, seed, temp)
        rows.append(dict(tag="D", system="dann", protocol="loco",
                         source=source_label(list(sources)), held_out_corpus=held or "",
                         target=target, seed=int(seed), held_out=bool(target not in sources),
                         **audit, test_macro_f1=m["macro_f1"], test_accuracy=m["accuracy"],
                         test_recall_class_0=m["recall_class_0"],
                         test_recall_class_1=m["recall_class_1"], temperature=temp,
                         ece=expected_calibration_error(clean.label_binary.values, softmax_np(logits)),
                         train_seconds=seconds, config_hash=CONFIG_HASH,
                         prediction_path=str(pth.relative_to(ROOT))))
        print("    -> %-12s F1=%.4f" % (DISPLAY[target], m["macro_f1"]))
    del model; free_gpu()
    return rows

d_rows = load_rows(D_RUNS)
D_DONE = {(str(r["source"]), int(r["seed"])) for r in d_rows} if d_rows else set()
jobs_D = [(src, held, sd) for src, held in LOCO_FOLDS for sd in DANN_SEEDS
          if (source_label(list(src)), sd) not in D_DONE]
print("Stage D pending jobs:", len(jobs_D))

if RUN_STAGE_D_DANN and jobs_D:
    try:
        per_job = None
        for i, (sources, held, seed) in enumerate(jobs_D):
            enforce_budget(len(jobs_D) - i, per_job)
            print("  [D|dann|hold out %s|seed %s]" % (DISPLAY[held], seed))
            rows = run_dann(sources, held, seed)
            per_job = rows[0]["train_seconds"] if per_job is None else 0.5 * (per_job + rows[0]["train_seconds"])
            d_rows = [r for r in d_rows
                      if (str(r.get("source")), int(r.get("seed", -1)))
                      != (source_label(list(sources)), int(seed))]
            d_rows.extend(rows)
            pd.DataFrame(d_rows).to_csv(D_RUNS, index=False)
    except Exception as exc:
        print("Stage D failed and was skipped (earlier stages are intact):", repr(exc))
elif not RUN_STAGE_D_DANN:
    print("Stage D disabled.")
else:
    print("Stage D already complete.")

dann_summary = pd.DataFrame()
if Path(D_RUNS).exists() and len(pd.read_csv(D_RUNS)):
    dd = pd.read_csv(D_RUNS)
    rows = []
    for (src, tgt), g in dd[dd.held_out].groupby(["source", "target"]):
        m, sd, lo, hi = tci(g.test_macro_f1)
        base = loco_summary[(loco_summary.system == "fgm") & (loco_summary.protocol == "loco") &
                            (loco_summary.target == tgt) & loco_summary.held_out] \
               if len(loco_summary) else pd.DataFrame()
        rows.append(dict(source=src, target=tgt, seeds=int(g.seed.nunique()),
                         dann_macro_f1=m, dann_std=sd, dann_ci95_lo=lo, dann_ci95_hi=hi,
                         loco_fgm_macro_f1=float(base.macro_f1_mean.iloc[0]) if len(base) else np.nan))
    dann_summary = pd.DataFrame(rows)
    dann_summary["dann_minus_loco"] = dann_summary.dann_macro_f1 - dann_summary.loco_fgm_macro_f1
    display(dann_summary.round(4))

Stage D pending jobs: 9
  [budget] spent 2.58 GPU-h ($2.06) | projected 3.33 GPU-h ($2.66) | cap $10.00
  [D|dann|hold out Ben-Sarc|seed 42]
    epoch 1 val macro-F1 0.8088
    epoch 2 val macro-F1 0.8162
    epoch 3 val macro-F1 0.8079
    epoch 4 val macro-F1 0.8200
    -> Ben-Sarc     F1=0.4854
    -> BanglaSarc   F1=0.9200
    -> BanglaSarc3  F1=0.7475
  [budget] spent 2.59 GPU-h ($2.07) | projected 2.72 GPU-h ($2.18) | cap $10.00
  [D|dann|hold out Ben-Sarc|seed 1]
    epoch 1 val macro-F1 0.7787
    epoch 2 val macro-F1 0.8082
    epoch 3 val macro-F1 0.8090
    epoch 4 val macro-F1 0.8075
    -> Ben-Sarc     F1=0.4849
    -> BanglaSarc   F1=0.9263
    -> BanglaSarc3  F1=0.7344
  [budget] spent 2.61 GPU-h ($2.09) | projected 2.72 GPU-h ($2.18) | cap $10.00
  [D|dann|hold out Ben-Sarc|seed 7]
    epoch 1 val macro-F1 0.8089
    epoch 2 val macro-F1 0.8123
    epoch 3 val macro-F1 0.8278
    epoch 4 val macro-F1 0.8188
    -> Ben-Sarc     F1=0.4656
    -> BanglaSarc   F1=0.9283
   

,source,target,seeds,dann_macro_f1,dann_std,dann_ci95_lo,dann_ci95_hi,loco_fgm_macro_f1,dann_minus_loco
0,banglasarc_binary+banglasarc3_binary,ben_sarc_binary,3,0.4786,0.0113,0.4505,0.5068,0.5087,-0.0300
1,ben_sarc_binary+banglasarc3_binary,banglasarc_binary,3,0.5998,0.0234,0.5416,0.6580,0.6207,-0.0209
2,ben_sarc_binary+banglasarc_binary,banglasarc3_binary,3,0.6248,0.0060,0.6100,0.6396,0.6386,-0.0139


---
## Stage E — Source-validation threshold recalibration

CPU-only, zero GPU cost. The single most dramatic failure in the previous manuscript was
BanglaSarc→BanglaSarc3 collapsing to 0.009 sarcastic-class recall. Argmax decoding on a shifted
distribution is a plausible contributor. Here we choose the decision threshold that maximises
**source validation** macro-F1 and apply it unchanged to every target, then report what it buys.

This closes the paper's worst failure with an attempted fix rather than a shrug, and the threshold
is selected without ever touching target data.

In [16]:
thr_rows = []
if RUN_STAGE_E_THRESHOLD:
    grid = np.linspace(0.05, 0.95, 91)
    for system in SYSTEMS:
        for source in CORPORA:
            for seed in SEEDS_10:
                vl = CKPT / f"valLogits_A_{system}_{source}_seed{seed}.npy"
                vg = CKPT / f"valGold_A_{system}_{source}_seed{seed}.csv"
                if not (vl.exists() and vg.exists()):
                    continue
                vprob = softmax_np(np.load(vl))[:, 1]
                vgold = pd.read_csv(vg)["gold"].values
                scores = [f1_score(vgold, (vprob >= t).astype(int), average="macro", zero_division=0)
                          for t in grid]
                t_star = float(grid[int(np.argmax(scores))])
                for target in CORPORA:
                    pth = (PRED / "19_singlesource_newseeds" /
                           f"19_A_{system}_{source}_seed{seed}_to_{target}.csv")
                    if not pth.exists():
                        continue
                    p = pd.read_csv(pth)
                    gold = p.gold_label.values
                    base = score(gold, p.pred_label.values)
                    recal = score(gold, (p.prob_1.values >= t_star).astype(int))
                    thr_rows.append(dict(
                        system=system, source=source, target=target, seed=int(seed),
                        in_domain=bool(source == target), threshold=t_star,
                        macro_f1_argmax=base["macro_f1"], macro_f1_recalibrated=recal["macro_f1"],
                        delta_macro_f1=recal["macro_f1"] - base["macro_f1"],
                        recall_class_1_argmax=base["recall_class_1"],
                        recall_class_1_recalibrated=recal["recall_class_1"],
                        delta_recall_class_1=recal["recall_class_1"] - base["recall_class_1"]))
if thr_rows:
    thr = pd.DataFrame(thr_rows)
    agg = (thr.groupby(["system", "source", "target", "in_domain"])
             .agg(seeds=("seed", "nunique"), threshold=("threshold", "mean"),
                  macro_f1_argmax=("macro_f1_argmax", "mean"),
                  macro_f1_recalibrated=("macro_f1_recalibrated", "mean"),
                  delta_macro_f1=("delta_macro_f1", "mean"),
                  recall_class_1_argmax=("recall_class_1_argmax", "mean"),
                  recall_class_1_recalibrated=("recall_class_1_recalibrated", "mean"))
             .reset_index())
    agg.to_csv(E_TAB, index=False)
    off = agg[~agg.in_domain]
    print("threshold recalibration on off-diagonal cells: mean delta macro-F1 = %+.4f"
          % off.delta_macro_f1.mean())
    display(off.round(4))
else:
    print("Stage E produced no rows (needs Stage A validation logits from this notebook).")

threshold recalibration on off-diagonal cells: mean delta macro-F1 = +0.0028


,system,source,target,in_domain,seeds,threshold,macro_f1_argmax,macro_f1_recalibrated,delta_macro_f1,recall_class_1_argmax,recall_class_1_recalibrated
1,fgm,banglasarc3_binary,banglasarc_binary,False,5,0.526,0.5987,0.5959,-0.0028,0.5566,0.5169
2,fgm,banglasarc3_binary,ben_sarc_binary,False,5,0.526,0.6640,0.6652,0.0011,0.7187,0.6927
3,fgm,banglasarc_binary,banglasarc3_binary,False,5,0.442,0.3361,0.3371,0.0010,0.0102,0.0113
5,fgm,banglasarc_binary,ben_sarc_binary,False,5,0.442,0.3456,0.3462,0.0006,0.0140,0.0148
6,fgm,ben_sarc_binary,banglasarc3_binary,False,5,0.412,0.6542,0.6581,0.0039,0.6238,0.6648
7,fgm,ben_sarc_binary,banglasarc_binary,False,5,0.412,0.5930,0.6048,0.0118,0.4422,0.4855
10,vanilla,banglasarc3_binary,banglasarc_binary,False,5,0.448,0.6143,0.6157,0.0014,0.5880,0.5867
11,vanilla,banglasarc3_binary,ben_sarc_binary,False,5,0.448,0.6781,0.6809,0.0029,0.7292,0.7276
12,vanilla,banglasarc_binary,banglasarc3_binary,False,5,0.486,0.3357,0.3349,-0.0008,0.0102,0.0097
14,vanilla,banglasarc_binary,ben_sarc_binary,False,5,0.486,0.3472,0.3476,0.0004,0.0162,0.0167


---
## Stage F — Near-duplicate sensitivity

CPU-only, zero GPU cost. Notebook 01 ran with `NEAR_DUP = False`, so the manuscript's
deduplication controls normalized *exact* matches only. That is honest but narrower than a referee
will want, and it is the weakest point in the current title.

The key observation is that **no retraining is required**. We build a stricter key that also
collapses punctuation, digits and character elongation, plus a character-n-gram TF-IDF cosine
neighbour search, and then **re-score the already-saved prediction files** on the surviving subset.
If the conclusions hold under the stricter filter, the deduplication claim is defended for the price
of a few CPU minutes.

In [17]:
neardup_audit, neardup_sens = pd.DataFrame(), pd.DataFrame()
if RUN_STAGE_F_NEARDUP:
    COS_THRESHOLD = 0.92
    rows = []
    all_text = pd.concat([DATA[c][s] for c in CORPORA for s in ("train", "val", "test")],
                         ignore_index=True)
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_features=200000)
    print("fitting character TF-IDF over %d documents ..." % len(all_text))
    X = vec.fit_transform(all_text.text.tolist())

    drop_keys = {}
    for source in CORPORA:
        src_near = set()
        for s in ("train", "val", "test"):
            src_near |= set(DATA[source][s]["near_key"])
        src_idx = all_text.index[all_text.corpus == source]
        Xs = X[src_idx]
        for target in CORPORA:
            if target == source:
                continue
            te = DATA[target]["test"]
            exact_removed = te["norm_key_nb18"].isin(
                set().union(*[set(DATA[source][s]["norm_key_nb18"]) for s in ("train", "val", "test")]))
            near_hit = te["near_key"].isin(src_near)
            te_idx = all_text.index[(all_text.corpus == target) &
                                    all_text.item_id.isin(te.item_id)]
            sims = cosine_similarity(X[te_idx], Xs, dense_output=False)
            max_sim = np.asarray(sims.max(axis=1).todense()).ravel() if sims.nnz else np.zeros(len(te))
            fuzzy_hit = pd.Series(max_sim >= COS_THRESHOLD, index=te.index)
            drop = exact_removed | near_hit | fuzzy_hit
            drop_keys[(source, target)] = set(te.loc[drop, "item_id"])
            rows.append(dict(source=source, target=target, n_target_test=len(te),
                             removed_exact=int(exact_removed.sum()),
                             removed_near_key_extra=int((near_hit & ~exact_removed).sum()),
                             removed_fuzzy_extra=int((fuzzy_hit & ~exact_removed & ~near_hit).sum()),
                             removed_total_strict=int(drop.sum()),
                             n_remaining_strict=int((~drop).sum()),
                             cosine_threshold=COS_THRESHOLD))
            print("  %-12s -> %-12s exact=%3d  +near=%3d  +fuzzy=%3d  remaining=%4d"
                  % (DISPLAY[source], DISPLAY[target], rows[-1]["removed_exact"],
                     rows[-1]["removed_near_key_extra"], rows[-1]["removed_fuzzy_extra"],
                     rows[-1]["n_remaining_strict"]))
    neardup_audit = pd.DataFrame(rows)
    neardup_audit.to_csv(F_AUDIT, index=False)

    # Re-score saved predictions on the strict subset. No retraining.
    sens = []
    for pdir in ("19_singlesource_newseeds", "18_multiseed_cross_corpus"):
        if not (PRED / pdir).exists():
            continue
        for pth in sorted((PRED / pdir).glob("*.csv")):
            p = pd.read_csv(pth)
            src, tgt = str(p.source.iloc[0]), str(p.target.iloc[0])
            if src == tgt or (src, tgt) not in drop_keys:
                continue
            keep = ~p.item_id.isin(drop_keys[(src, tgt)])
            if keep.sum() < 30:
                continue
            base = f1_score(p.gold_label, p.pred_label, average="macro", zero_division=0)
            strict = f1_score(p.loc[keep, "gold_label"], p.loc[keep, "pred_label"],
                              average="macro", zero_division=0)
            sens.append(dict(system=str(p.system.iloc[0]), source=src, target=tgt,
                             seed=int(p.seed.iloc[0]), n_exact_filtered=len(p),
                             n_strict_filtered=int(keep.sum()),
                             macro_f1_exact_filter=base, macro_f1_strict_filter=strict,
                             delta=strict - base))
    if sens:
        sdf = pd.DataFrame(sens)
        neardup_sens = (sdf.groupby(["system", "source", "target"])
                          .agg(seeds=("seed", "nunique"),
                               n_exact=("n_exact_filtered", "first"),
                               n_strict=("n_strict_filtered", "first"),
                               macro_f1_exact_filter=("macro_f1_exact_filter", "mean"),
                               macro_f1_strict_filter=("macro_f1_strict_filter", "mean"),
                               delta=("delta", "mean")).reset_index())
        neardup_sens.to_csv(F_SENS, index=False)
        print("\\nmean |delta| under the strict filter: %.4f (max %.4f)"
              % (neardup_sens.delta.abs().mean(), neardup_sens.delta.abs().max()))
        display(neardup_sens.round(4))
    else:
        print("\\nNo Stage A prediction files yet; rerun Stage F after Stage A completes.")

fitting character TF-IDF over 38168 documents ...
  Ben-Sarc     -> BanglaSarc   exact=  0  +near=  2  +fuzzy=  0  remaining= 462
  Ben-Sarc     -> BanglaSarc3  exact=  0  +near=  0  +fuzzy=  0  remaining= 791
  BanglaSarc   -> Ben-Sarc     exact=  0  +near=  0  +fuzzy=  0  remaining=2563
  BanglaSarc   -> BanglaSarc3  exact= 29  +near= 61  +fuzzy=  6  remaining= 695
  BanglaSarc3  -> Ben-Sarc     exact=  0  +near=  2  +fuzzy=  0  remaining=2561
  BanglaSarc3  -> BanglaSarc   exact= 28  +near= 59  +fuzzy=  5  remaining= 372
\nmean |delta| under the strict filter: 0.0019 (max 0.0091)


,system,source,target,seeds,n_exact,n_strict,macro_f1_exact_filter,macro_f1_strict_filter,delta
0,fgm,banglasarc3_binary,banglasarc_binary,10,436,372,0.6113,0.6106,-0.0007
1,fgm,banglasarc3_binary,ben_sarc_binary,10,2563,2561,0.6670,0.6668,-0.0002
2,fgm,banglasarc_binary,banglasarc3_binary,10,762,695,0.3351,0.3260,-0.0091
3,fgm,banglasarc_binary,ben_sarc_binary,10,2563,2563,0.3458,0.3458,0.0000
4,fgm,ben_sarc_binary,banglasarc3_binary,10,791,791,0.6516,0.6516,0.0000
5,fgm,ben_sarc_binary,banglasarc_binary,10,464,462,0.5943,0.5934,-0.0009
6,vanilla,banglasarc3_binary,banglasarc_binary,10,436,372,0.6279,0.6258,-0.0020
7,vanilla,banglasarc3_binary,ben_sarc_binary,10,2563,2561,0.6728,0.6726,-0.0002
8,vanilla,banglasarc_binary,banglasarc3_binary,10,762,695,0.3365,0.3274,-0.0091
9,vanilla,banglasarc_binary,ben_sarc_binary,10,2563,2563,0.3467,0.3467,0.0000


---
## Stage G — Local open-weight LLM baseline (optional, OFF by default)

Set `RUN_STAGE_G_LLM = True` to enable. This runs a locally hosted instruction model, so no comment
text leaves the machine and the dataset-licensing objection to the previous prompted-API experiment
disappears entirely.

A 7B model in bfloat16 needs roughly 15 GB and fits on a 24 GB card. `LLM_MAX_EVAL` subsamples each
corpus stratified by label to keep inference bounded; set it to a large number to score everything.

Enable this only after Stages A–C have finished and their tables are on disk — the download is
large and inference is the slowest part of the notebook.

In [18]:
llm_table = pd.DataFrame()
if RUN_STAGE_G_LLM:
    try:
        from transformers import AutoModelForCausalLM
        device = torch.device("cuda")
        print("loading", LLM_MODEL_NAME, "...")
        ltok = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
        lmodel = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL_NAME, torch_dtype=torch.bfloat16 if HAS_BF16 else torch.float16,
            device_map="auto")
        lmodel.eval()
        if ltok.pad_token_id is None:
            ltok.pad_token = ltok.eos_token
        ltok.padding_side = "left"

        SYS = ("You are an expert annotator of Bengali (Bangla) social media comments. "
               "Decide whether a comment is sarcastic. Answer with exactly one word: "
               "either SARCASTIC or NOT_SARCASTIC. Output nothing else.")

        def build_prompt(text, shots):
            msgs = [{"role": "system", "content": SYS}]
            for t, y in shots:
                msgs.append({"role": "user", "content": "Comment: " + t})
                msgs.append({"role": "assistant",
                             "content": "SARCASTIC" if y == 1 else "NOT_SARCASTIC"})
            msgs.append({"role": "user", "content": "Comment: " + text})
            return ltok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

        def parse(out):
            u = out.strip().upper()
            if "NOT_SARCASTIC" in u or "NOT SARCASTIC" in u:
                return 0
            if "SARCASTIC" in u:
                return 1
            return -1

        @torch.no_grad()
        def run_llm(corpus, n_shots, seed=42, batch=16):
            te = DATA[corpus]["test"]
            if len(te) > LLM_MAX_EVAL:
                te = (te.groupby("label_binary", group_keys=False)
                        .apply(lambda g: g.sample(n=max(1, int(round(LLM_MAX_EVAL * len(g) / len(te)))),
                                                  random_state=seed))
                        .reset_index(drop=True))
            shots = []
            if n_shots:
                tr = DATA[corpus]["train"]
                rng = np.random.default_rng(seed)
                for label in (0, 1):
                    g = tr[tr.label_binary == label]
                    for i in rng.choice(g.index.values, size=n_shots, replace=False):
                        shots.append((str(tr.loc[i, "text"]), int(label)))
                rng.shuffle(shots)
            prompts = [build_prompt(t, shots) for t in te.text.tolist()]
            preds, unresolved = [], 0
            for i in range(0, len(prompts), batch):
                enc = ltok(prompts[i:i + batch], return_tensors="pt", padding=True,
                           truncation=True, max_length=2048).to(device)
                gen = lmodel.generate(**enc, max_new_tokens=8, do_sample=False,
                                      pad_token_id=ltok.pad_token_id)
                for j in range(gen.shape[0]):
                    txt = ltok.decode(gen[j][enc["input_ids"].shape[1]:], skip_special_tokens=True)
                    p = parse(txt)
                    if p < 0:
                        unresolved += 1
                        p = 0
                    preds.append(p)
                if (i // batch) % 10 == 0:
                    print("    %d/%d" % (i, len(prompts)), end="\\r")
            m = score(te.label_binary.values, np.asarray(preds))
            return dict(model=LLM_MODEL_NAME, corpus=corpus, setting=("few_shot" if n_shots else "zero_shot"),
                        fewshot_per_class=n_shots, n_eval=len(te), macro_f1=m["macro_f1"],
                        accuracy=m["accuracy"], recall_class_0=m["recall_class_0"],
                        recall_class_1=m["recall_class_1"],
                        unresolved_rate=unresolved / max(1, len(preds)),
                        hosted="local", prompt_lang="en", config_hash=CONFIG_HASH)

        out = []
        for corpus in CORPORA:
            for n_shots in (0, 3):
                print("  %s | %d-shot" % (DISPLAY[corpus], n_shots))
                r = run_llm(corpus, n_shots)
                out.append(r)
                print("    macro-F1 = %.4f (n=%d, unresolved=%.3f)"
                      % (r["macro_f1"], r["n_eval"], r["unresolved_rate"]))
        llm_table = pd.DataFrame(out)
        # Triangulation: fine-tuning advantage over a model that never saw any training split
        ft = {c: ceiling_for("fgm", c) for c in CORPORA}
        llm_table["finetuned_macro_f1"] = llm_table.corpus.map(ft)
        llm_table["finetuning_advantage"] = llm_table.finetuned_macro_f1 - llm_table.macro_f1
        llm_table.to_csv(G_TAB, index=False)
        display(llm_table.round(4))
        del lmodel; free_gpu()
    except Exception as exc:
        print("Stage G failed and was skipped:", repr(exc))
else:
    print("Stage G disabled (RUN_STAGE_G_LLM = False).")

Stage G disabled (RUN_STAGE_G_LLM = False).


---
## Figures

Vector PDF plus 600-dpi PNG, sized for a Springer single-column layout. Every figure is written to
`04_outputs/finalized_outputs/figures/`.

In [19]:
C_ = dict(blue="#0072B2", orange="#E69F00", green="#009E73", vermillion="#D55E00",
          purple="#CC79A7", sky="#56B4E9", yellow="#F0E442", gray="#777777",
          light="#D9E2E8", dark="#263238")
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 600, "savefig.bbox": "tight",
    "font.family": "serif", "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix", "font.size": 8.2, "axes.titlesize": 9.0,
    "axes.labelsize": 8.2, "axes.linewidth": 0.8, "axes.edgecolor": "#333333",
    "xtick.labelsize": 7.2, "ytick.labelsize": 7.2, "legend.fontsize": 7.0,
    "figure.facecolor": "white", "axes.facecolor": "white"})
IN1, IN2 = 3.5, 7.16
generated_figures = []

def savefig(fig, stem):
    for ext in ("pdf", "png"):
        fig.savefig(FF / ("%s.%s" % (stem, ext)))
    generated_figures.append(stem)
    plt.close(fig)
    print("figure:", stem)

# F19-1  Label-efficiency recovery curves  (the headline figure)
if len(labeleff_summary):
    fig, axes = plt.subplots(1, 3, figsize=(IN2, 2.5), sharey=True)
    for ax, target in zip(axes, CORPORA):
        d = labeleff_summary[labeleff_summary.target == target]
        ceiling = float(d.target_trained_ceiling.iloc[0])
        for colour, src in zip((C_["blue"], C_["vermillion"]),
                               [c for c in CORPORA if c != target]):
            g = d[d.source == src].sort_values("k")
            if not len(g):
                continue
            ax.plot(g.k, g.macro_f1_mean, marker="o", ms=3, lw=1.2, color=colour,
                    label="from " + DISPLAY[src])
            ax.fill_between(g.k, g.macro_f1_ci95_lo, g.macro_f1_ci95_hi, color=colour, alpha=0.15)
        ax.axhline(ceiling, ls="--", lw=1.0, color=C_["dark"])
        ax.axhline(0.90 * ceiling, ls=":", lw=0.9, color=C_["gray"])
        ax.set_xscale("symlog", linthresh=25)
        ax.set_xticks([0, 25, 100, 500, 1000]); ax.set_xticklabels(["0", "25", "100", "500", "1k"])
        ax.set_title("Target: " + DISPLAY[target]); ax.set_xlabel("Labelled target examples (k)")
        ax.legend(frameon=False, loc="lower right")
    axes[0].set_ylabel("Macro-F1")
    fig.suptitle("A small labelled target sample recovers most of the transfer gap", y=1.04)
    savefig(fig, "F19_label_efficiency")

# F19-2  LOCO vs single-source transfer vs target-trained ceiling
if len(pooled_vs_single):
    d = pooled_vs_single[pooled_vs_single.system == "fgm"]
    if len(d):
        fig, ax = plt.subplots(figsize=(IN1, 2.7))
        x = np.arange(len(d)); w = 0.27
        ax.bar(x - w, d.mean_single_source_transfer, w, color=C_["vermillion"], label="Single-source transfer")
        ax.bar(x, d.loco_two_source, w, color=C_["blue"], label="Leave-one-corpus-out")
        ax.bar(x + w, d.target_trained_ceiling, w, color=C_["gray"], label="Target-trained ceiling")
        ax.set_xticks(x); ax.set_xticklabels([DISPLAY[t] for t in d.target], rotation=20, ha="right")
        ax.set_ylabel("Macro-F1"); ax.set_ylim(0, 1.05)
        ax.set_title("Multi-source training partially closes the transfer gap")
        ax.legend(frameon=False, ncol=1, loc="upper left")
        savefig(fig, "F19_loco_vs_single_source")

# F19-3  Ten-seed stability on the three diagonals
if len(merged):
    fig, axes = plt.subplots(1, 3, figsize=(IN2, 2.4), sharey=False)
    for ax, corpus in zip(axes, CORPORA):
        seeds_axis = []
        for colour, sysname in ((C_["gray"], "vanilla"), (C_["blue"], "fgm")):
            g = (merged[(merged.system == sysname) & (merged.source == corpus) &
                        (merged.target == corpus)].sort_values("seed"))
            if not len(g):
                continue
            seeds_axis = list(g.seed)
            ax.plot(range(len(g)), g.test_macro_f1, marker="o", ms=3, lw=1.0,
                    color=colour, label=sysname.upper())
        ax.set_xticks(range(len(seeds_axis)))
        ax.set_xticklabels([str(s) for s in seeds_axis], rotation=60, fontsize=5.5)
        ax.set_title(DISPLAY[corpus]); ax.set_xlabel("Seed")
        ax.legend(frameon=False)
    axes[0].set_ylabel("In-domain macro-F1")
    fig.suptitle("Matched ten-seed stability", y=1.05)
    savefig(fig, "F19_seed_stability_10")

# F19-4  FGM minus vanilla across all cells, ten seeds
if len(tests10):
    d = tests10.sort_values("delta_mean").reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(IN1, 3.0))
    y = np.arange(len(d))
    ax.errorbar(d.delta_mean, y,
                xerr=[d.delta_mean - d.delta_ci95_lo, d.delta_ci95_hi - d.delta_mean],
                fmt="o", ms=3.5, lw=1.0, capsize=2, color=C_["blue"])
    ax.axvline(0, ls="--", lw=0.9, color=C_["dark"])
    ax.set_yticks(y)
    ax.set_yticklabels(["%s to %s" % (DISPLAY[s], DISPLAY[t]) for s, t in zip(d.source, d.target)],
                       fontsize=6.3)
    ax.set_xlabel("FGM minus vanilla macro-F1 (matched seeds)")
    ax.set_title("Adversarial fine-tuning effect, ten seeds")
    savefig(fig, "F19_fgm_vs_vanilla_10seed")

# F19-5  Sarcastic-class recall before and after threshold recalibration
if Path(E_TAB).exists():
    d = pd.read_csv(E_TAB)
    d = d[(~d.in_domain) & (d.system == "fgm")].reset_index(drop=True)
    if len(d):
        fig, ax = plt.subplots(figsize=(IN1, 2.7))
        x = np.arange(len(d)); w = 0.38
        ax.bar(x - w / 2, d.recall_class_1_argmax, w, color=C_["vermillion"], label="Argmax")
        ax.bar(x + w / 2, d.recall_class_1_recalibrated, w, color=C_["green"],
               label="Source-validation threshold")
        ax.set_xticks(x)
        ax.set_xticklabels(["%s to %s" % (DISPLAY[s][:6], DISPLAY[t][:6])
                            for s, t in zip(d.source, d.target)], rotation=35, ha="right", fontsize=6)
        ax.set_ylabel("Sarcastic-class recall"); ax.legend(frameon=False)
        ax.set_title("Threshold recalibration and the sarcastic-class collapse")
        savefig(fig, "F19_threshold_recalibration")

# F19-6  Near-duplicate sensitivity
if len(neardup_sens):
    d = neardup_sens[neardup_sens.system == "fgm"].reset_index(drop=True)
    if len(d):
        fig, ax = plt.subplots(figsize=(IN1, 2.6))
        x = np.arange(len(d)); w = 0.38
        ax.bar(x - w / 2, d.macro_f1_exact_filter, w, color=C_["blue"], label="Exact-match filter")
        ax.bar(x + w / 2, d.macro_f1_strict_filter, w, color=C_["orange"], label="Strict near-duplicate filter")
        ax.set_xticks(x)
        ax.set_xticklabels(["%s to %s" % (DISPLAY[s][:6], DISPLAY[t][:6])
                            for s, t in zip(d.source, d.target)], rotation=35, ha="right", fontsize=6)
        ax.set_ylabel("Transfer macro-F1"); ax.legend(frameon=False)
        ax.set_title("Conclusions under a stricter duplicate filter")
        savefig(fig, "F19_near_duplicate_sensitivity")

# F19-7  DANN comparison
if len(dann_summary):
    d = dann_summary.dropna(subset=["loco_fgm_macro_f1"]).reset_index(drop=True)
    if len(d):
        fig, ax = plt.subplots(figsize=(IN1, 2.6))
        x = np.arange(len(d)); w = 0.38
        ax.bar(x - w / 2, d.loco_fgm_macro_f1, w, color=C_["blue"], label="Pooled LOCO (FGM)")
        ax.bar(x + w / 2, d.dann_macro_f1, w, color=C_["purple"], label="DANN")
        ax.set_xticks(x); ax.set_xticklabels([DISPLAY[t] for t in d.target], rotation=20, ha="right")
        ax.set_ylabel("Held-out macro-F1"); ax.legend(frameon=False)
        ax.set_title("Domain-adversarial training on held-out corpora")
        savefig(fig, "F19_dann_vs_pooled")

# F19-8  LLM triangulation
if len(llm_table):
    d = llm_table[llm_table.setting == "few_shot"].reset_index(drop=True)
    if len(d):
        fig, ax = plt.subplots(figsize=(IN1, 2.6))
        x = np.arange(len(d)); w = 0.38
        ax.bar(x - w / 2, d.macro_f1, w, color=C_["orange"], label="Local LLM, few-shot")
        ax.bar(x + w / 2, d.finetuned_macro_f1, w, color=C_["blue"], label="Fine-tuned BanglaBERT")
        for i, v in enumerate(d.finetuning_advantage):
            ax.text(i, max(d.macro_f1[i], d.finetuned_macro_f1[i]) + 0.02,
                    "%+.2f" % v, ha="center", fontsize=6.2)
        ax.set_xticks(x); ax.set_xticklabels([DISPLAY[c] for c in d.corpus], rotation=20, ha="right")
        ax.set_ylim(0, 1.12); ax.set_ylabel("Macro-F1"); ax.legend(frameon=False, loc="lower right")
        ax.set_title("Fine-tuning advantage over an unexposed model")
        savefig(fig, "F19_llm_triangulation")

print("\\nfigures written:", len(generated_figures))

figure: F19_label_efficiency
figure: F19_loco_vs_single_source
figure: F19_seed_stability_10
figure: F19_fgm_vs_vanilla_10seed
figure: F19_threshold_recalibration
figure: F19_near_duplicate_sensitivity
figure: F19_dann_vs_pooled
\nfigures written: 7


---
## Claim ledger and artifact manifest

Every number the manuscript may cite is written to `19_claim_ledger.csv` with its source file,
configuration hash, and confirmatory/exploratory status. Fill the `manuscript_location` column as
you write, then verify that no abstract or conclusion number lacks a ledger row.

In [20]:
ledger = []
def claim(cid, metric, value, source_file, status="confirmatory", note=""):
    ledger.append(dict(claim_id=cid, metric=metric, value=value,
                       source_file=str(Path(source_file).relative_to(ROOT))
                       if Path(source_file).exists() else str(source_file),
                       config_hash=CONFIG_HASH, git_commit=RUN_META["git"]["commit"],
                       status=status, manuscript_location="", note=note))

if len(summary10):
    for sysname in ("fgm", "vanilla"):
        d = summary10[summary10.system == sysname]
        claim("C1_%s_indomain_mean" % sysname, "macro_f1", round(float(d[d.in_domain].macro_f1_mean.mean()), 4),
              A_SUMM, note="mean of three diagonal cells, ten seeds")
        claim("C2_%s_transfer_mean" % sysname, "macro_f1", round(float(d[~d.in_domain].macro_f1_mean.mean()), 4),
              A_SUMM, note="mean of six off-diagonal cells, ten seeds")
if len(tests10):
    n = int(tests10.n_seeds.min())
    claim("C3_min_attainable_p", "p", round(2 / 2 ** n, 5), A_TESTS,
          note="minimum two-sided exact sign-flip p with %d matched seeds" % n)
    claim("C4_n_significant_holm", "count", int(tests10.significant_holm_0_05.sum()), A_TESTS)
if len(pooled_vs_single):
    fg = pooled_vs_single[pooled_vs_single.system == "fgm"]
    if len(fg):
        claim("C5_loco_gain", "delta_macro_f1", round(float(fg.gain_vs_mean_single.mean()), 4), B_COMP,
              note="LOCO minus mean single-source transfer, averaged over targets")
        claim("C6_retention_single", "retention", round(float(fg.retention_single_mean.mean()), 4), B_COMP)
        claim("C7_retention_loco", "retention", round(float(fg.retention_loco.mean()), 4), B_COMP)
if len(k_to_reach):
    m90 = k_to_reach["k_for_90_pct"].dropna()
    if len(m90):
        claim("C8_median_k_for_90pct", "k", int(m90.median()), C_KTO,
              note="median labelled target examples to reach 90% of target-trained macro-F1")
        claim("C9_pairs_reaching_90pct", "count", "%d/%d" % (len(m90), len(k_to_reach)), C_KTO)
if len(neardup_sens):
    claim("C10_neardup_max_abs_delta", "delta_macro_f1", round(float(neardup_sens.delta.abs().max()), 4),
          F_SENS, note="largest change under the strict near-duplicate filter")
if Path(E_TAB).exists():
    d = pd.read_csv(E_TAB); off = d[~d.in_domain]
    claim("C11_threshold_gain_offdiag", "delta_macro_f1", round(float(off.delta_macro_f1.mean()), 4), E_TAB)
if len(dann_summary):
    claim("C12_dann_minus_loco", "delta_macro_f1", round(float(dann_summary.dann_minus_loco.mean()), 4),
          D_RUNS, note="DANN minus pooled LOCO on held-out corpora")
if len(llm_table):
    for r in llm_table[llm_table.setting == "few_shot"].itertuples():
        claim("C13_ft_advantage_%s" % r.corpus, "delta_macro_f1", round(float(r.finetuning_advantage), 4),
              G_TAB, note="fine-tuned minus local LLM few-shot")

ledger_df = pd.DataFrame(ledger)
ledger_df.to_csv(LEDGER, index=False)
display(ledger_df[["claim_id", "metric", "value", "status", "note"]])

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

manifest = {"generated_utc": datetime.now(timezone.utc).isoformat(),
            "config_hash": CONFIG_HASH, "git": RUN_META["git"], "files": {}}
for p in sorted(list(FT.glob("19_*")) + list(FF.glob("F19_*"))):
    manifest["files"][str(p.relative_to(ROOT))] = dict(bytes=p.stat().st_size, sha256=sha256_file(p))
json.dump(manifest, open(MANIFEST, "w"), indent=2)
print("\\nmanifest entries:", len(manifest["files"]))

,claim_id,metric,value,status,note
0,C1_fgm_indomain_mean,macro_f1,0.8472,confirmatory,"mean of three diagonal cells, ten seeds"
1,C2_fgm_transfer_mean,macro_f1,0.5342,confirmatory,"mean of six off-diagonal cells, ten seeds"
2,C1_vanilla_indomain_mean,macro_f1,0.8417,confirmatory,"mean of three diagonal cells, ten seeds"
3,C2_vanilla_transfer_mean,macro_f1,0.5357,confirmatory,"mean of six off-diagonal cells, ten seeds"
4,C3_min_attainable_p,p,0.00195,confirmatory,minimum two-sided exact sign-flip p with 10 ma...
5,C4_n_significant_holm,count,0,confirmatory,
6,C5_loco_gain,delta_macro_f1,0.0552,confirmatory,"LOCO minus mean single-source transfer, averag..."
7,C6_retention_single,retention,0.6319,confirmatory,
8,C7_retention_loco,retention,0.7026,confirmatory,
9,C8_median_k_for_90pct,k,100,confirmatory,median labelled target examples to reach 90% o...


\nmanifest entries: 27


In [21]:
# Final report
elapsed = time.time() - NOTEBOOK_START
gpu_h = _gpu_seconds_spent[0] / 3600.0
print("=" * 74)
print("NOTEBOOK 19 COMPLETE")
print("=" * 74)
print("config hash        :", CONFIG_HASH)
print("git commit         :", RUN_META["git"]["commit"], "| dirty:", RUN_META["git"]["dirty"])
print("wall time          : %.1f min" % (elapsed / 60))
print("charged GPU time   : %.2f h  (approx $%.2f at $%.2f/h)"
      % (gpu_h, gpu_h * HOURLY_RATE_USD, HOURLY_RATE_USD))
print("-" * 74)
for label, obj in [("10-seed runs", merged), ("10-seed summary", summary10),
                   ("FGM vs vanilla tests", tests10), ("LOCO/pooled summary", loco_summary),
                   ("pooled vs single", pooled_vs_single), ("label-efficiency", labeleff_summary),
                   ("k-to-reach", k_to_reach), ("DANN", dann_summary),
                   ("near-dup sensitivity", neardup_sens), ("LLM baselines", llm_table),
                   ("claim ledger", ledger_df)]:
    print("  %-24s %s rows" % (label, len(obj) if obj is not None else 0))
print("-" * 74)
print("tables  ->", FT.relative_to(ROOT))
print("figures ->", FF.relative_to(ROOT), "(%d figures)" % len(generated_figures))
print("=" * 74)
for line in [
    "",
    "NEXT STEPS",
    "  1. Download 04_outputs/finalized_outputs/ (tables + figures) and 19_MANIFEST_sha256.json.",
    "  2. Terminate the pod immediately. Per-second billing punishes idle time.",
    "  3. Send the tables back so the manuscript numbers can be filled in.",
    "  4. Record the CONFIG HASH and git commit above in the manuscript Limitations section.",
]:
    print(line)

NOTEBOOK 19 COMPLETE
config hash        : 6970de89ef36b0ad
git commit         : a64ee91ba5ee8b30b94f66437afb4228bce5414b | dirty: True
wall time          : 185.9 min
charged GPU time   : 2.87 h  (approx $2.29 at $0.80/h)
--------------------------------------------------------------------------
  10-seed runs             180 rows
  10-seed summary          18 rows
  FGM vs vanilla tests     9 rows
  LOCO/pooled summary      24 rows
  pooled vs single         6 rows
  label-efficiency         42 rows
  k-to-reach               6 rows
  DANN                     3 rows
  near-dup sensitivity     12 rows
  LLM baselines            0 rows
  claim ledger             14 rows
--------------------------------------------------------------------------
tables  -> 04_outputs/finalized_outputs/tables
figures -> 04_outputs/finalized_outputs/figures (7 figures)

NEXT STEPS
  1. Download 04_outputs/finalized_outputs/ (tables + figures) and 19_MANIFEST_sha256.json.
  2. Terminate the pod immediately. P